# 📈 06 · Linear regression and genome-wide association
### From one SNP to the whole genome: maximum likelihood, least squares, the normal equations and a prostate-cancer study

**BiologyToML** · Level 1 · 02-620 core · Part B — Supervised learning · Module 06 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/06_linear_regression_gwas.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.stats`), pandas, Matplotlib, scikit-learn (for checking only) |
| **Prerequisites** | Modules 03–05 (vectors and matrices, derivatives and gradients, likelihood and MLE). No statistics course needed. |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **02-620 topics covered** | Regression, part 1: genome-wide association studies and Manhattan plots, discrete vs real-valued phenotypes, univariate linear regression with Gaussian noise, maximum conditional likelihood = least squares, multivariate regression and the normal equations, the prostate-cancer example, limitations of the MLE when $N<J$ |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data in this notebook are SIMULATED by code.** The GWAS cohorts are Hardy–Weinberg genotypes with known causal SNPs, so that we can check which SNPs a method finds; the prostate-cancer table is a simulated look-alike of the Stamey *et al.* (1989) study of 97 men. The seven short DNA sequences in Section 1 are a classroom example of the 02-620 lecture.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** how a genome-wide association study (GWAS) turns genotypes and a phenotype into a regression problem (the vector $\mathbf y$, the matrix $\mathbf X$, the 0/1/2 coding) and **read** a Manhattan plot.
2. **Write down** the univariate linear-regression model $y = w_0 + wx + \epsilon$ with Gaussian noise and **name** its parts (response, predictor, intercept, regression coefficient, noise).
3. **Derive**, step by step, that maximising the conditional likelihood is the same as minimising the squared error, and **solve** for $w_0$ and $w$ in closed form.
4. **Compute** a $t$-statistic and $p$-value for every SNP, **draw** a Manhattan plot with a Bonferroni threshold, and **explain** why a GWAS hit is an association, not proof of causation.
5. **Derive** the normal equations $\mathbf X^\top\mathbf X\mathbf w = \mathbf X^\top\mathbf y$ with matrix calculus and **implement** multivariate least squares with `np.linalg.solve`, checked against scikit-learn.
6. **Fit** the prostate-cancer model of the lecture (97 men, 8 clinical measures) on simulated data shaped like the original study, **compare** the coefficients with those the original study gave (ESL Table 3.2), and **interpret** them with care.
7. **Demonstrate** why least squares breaks down when there are more SNPs than people ($N<J$) — the motivation for Module 07.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit | 5 min |
| 1 · GWAS in one picture | SNPs, genotype coding, family studies vs GWAS, Manhattan plots, types of phenotype | 15 min |
| 2 · A simulated GWAS cohort | 500 people × 2,000 SNPs with 8 known causal SNPs | 10 min |
| 3 · One SNP at a time: the model | $y=w_0+wx+\epsilon$, Gaussian noise, the two-step recipe, terminology | 15 min |
| 4 · Learning $w_0, w$: likelihood = least squares | full derivation, closed form, numerical checks | 25 min |
| 5 · Scanning the genome | standard errors, $t$, $p$-values, Manhattan plot, confounding | 20 min |
| 6 · All SNPs at once | augmented data, normal equations, cost | 25 min |
| 7 · A clinical example: prostate cancer | the original study's coefficients, a simulated look-alike, test error, interpretation | 20 min |
| 8 · Limitations of MLE | $N<J$: singular $\mathbf X^\top\mathbf X$, perfect but useless fits | 10 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 5 min |

## 🧫 The biological question

Your DNA differs from a stranger's at roughly one position in every thousand. Most of these differences are
**single-nucleotide polymorphisms (SNPs)**: places where some people carry, say, an `A` and others a `T`. A few SNPs
change a protein or how much of it is made — and thereby your cholesterol level, your risk of breast cancer, or how
well you digest milk as an adult (*lactase persistence*: carriers of certain variants near the lactase gene keep
digesting milk sugar after childhood). Consumer-genomics companies such as 23andMe compare a customer's genotypes with a
database of variants, each annotated with the traits and diseases it has been linked to — and anyone can look up the same
annotations in public genome browsers. Every one of those annotations rests on a study that asked: ***which positions in
the genome are associated with this trait?***

For rare diseases that run in families (Huntington's disease, cystic fibrosis), geneticists traced the disease through
pedigrees. For **common** diseases and continuous traits — blood pressure, insulin level, the expression level of a gene —
the workhorse since about 2005 has been the **genome-wide association study (GWAS)**: genotype thousands of *unrelated*
people at hundreds of thousands of SNPs, measure the trait, and test every SNP. A breast-cancer GWAS published in 2010,
for example, analysed 582,886 SNPs in 3,659 cases and 4,897 controls (Section 1 draws what its result looks like).

The statistical engine of a GWAS of a continuous trait is **linear regression**, the first machine-learning method of
02-620. The lecture's roadmap, which this notebook and the next follow:

1. **univariate regression** (one SNP at a time) learned by maximum likelihood — Sections 3–5;
2. **multivariate regression** (all SNPs at the same time) learned by maximum likelihood — Sections 6–8;
3. **MAP estimation**: ridge regression and sparse (lasso) regression — [Module 07](07_ridge_lasso_sparse_regression.ipynb).

You will build regression from first principles — probability model, likelihood, calculus — run a GWAS on a simulated
cohort, and fit the same machinery to a clinical problem: predicting prostate-specific antigen (PSA) levels in 97 men with
prostate cancer.

> 🧪 **Why simulated data?** The repository stores no data sets, so every data set in this course is generated by code
> in the notebook itself. Simulation has a second, bigger advantage: we *know the truth* — which SNPs are causal, what the
> real coefficients are — so we can check whether a method found it, something a real study never allows.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_prostate_clinical` generates a table shaped like the prostate-cancer study of Section 7; `simulate_genotypes` and `simulate_quantitative_trait` create a GWAS cohort whose causal SNPs we know; `standardize` z-scores columns. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd

def simulate_prostate_clinical(seed: int = 1989) -> pd.DataFrame:
    """Synthetic clinical data shaped like the prostate-cancer study of Stamey et al. (1989).

    97 men about to undergo radical prostatectomy; 8 clinical measurements — log cancer volume
    (``lcavol``), log prostate weight (``lweight``), ``age``, log benign prostatic hyperplasia
    amount (``lbph``), seminal-vesicle invasion (``svi``, 0/1), log capsular penetration
    (``lcp``), Gleason score (``gleason``) and % Gleason scores 4 or 5 (``pgg45``) — and the
    outcome ``lpsa`` = log(prostate-specific antigen).  Marginals and correlations mimic the
    original data (``lbph`` and ``lcp`` have a floor at log 0.25 = −1.386); ``lpsa`` is generated
    from the standardised predictors with the least-squares coefficients reported for the
    original study by Hastie, Tibshirani & Friedman (2009, Table 3.2) plus noise (sd 0.71).
    The boolean ``train`` marks a 67/30 training/test split.
    """
    rng = np.random.default_rng(seed)
    n = 97
    names = ["lcavol", "lweight", "age", "lbph", "svi", "lcp", "gleason", "pgg45"]
    R = np.array([[1.00, 0.29, 0.19, 0.01, 0.56, 0.66, 0.49, 0.50],         # rank correlations of the
                  [0.29, 1.00, 0.40, 0.49, 0.17, 0.18, 0.14, 0.17],         # original study (rounded)
                  [0.19, 0.40, 1.00, 0.34, 0.13, 0.15, 0.25, 0.27],
                  [0.01, 0.49, 0.34, 1.00, -0.09, 0.02, 0.11, 0.10],
                  [0.56, 0.17, 0.13, -0.09, 1.00, 0.62, 0.38, 0.48],
                  [0.66, 0.18, 0.15, 0.02, 0.62, 1.00, 0.58, 0.66],
                  [0.49, 0.14, 0.25, 0.11, 0.38, 0.58, 1.00, 0.87],
                  [0.50, 0.17, 0.27, 0.10, 0.48, 0.66, 0.87, 1.00]])
    w, V = np.linalg.eigh(R)
    R = V @ np.diag(np.clip(w, 0.02, None)) @ V.T                        # make it positive definite
    Z = rng.multivariate_normal(np.zeros(8), R, n)
    from scipy.stats import norm
    U = norm.cdf(Z)
    floor = -1.38629
    df = pd.DataFrame({
        "lcavol": 1.35 + 1.18 * Z[:, 0],
        "lweight": 3.63 + 0.43 * Z[:, 1],
        "age": np.clip(np.round(63.9 + 7.4 * Z[:, 2]), 41, 79).astype(int),
        "lbph": np.where(U[:, 3] < 0.44, floor, np.clip(1.28 + 0.77 * norm.ppf((U[:, 3] - 0.44) / 0.56), -0.8, 2.4)),
        "svi": (U[:, 4] > 0.78).astype(int),
        "lcp": np.where(U[:, 5] < 0.46, floor, np.clip(0.87 + 1.13 * norm.ppf((U[:, 5] - 0.46) / 0.54), -0.8, 2.9)),
        "gleason": np.searchsorted([0.36, 0.94, 0.95], U[:, 6]) + 6,
    })
    df["pgg45"] = np.where(df["gleason"] == 6, 0, np.clip(5 * np.round((38 + 27 * Z[:, 7]) / 5), 5, 100)).astype(int)
    zs = (df - df.mean()) / df.std()
    coef = dict(lcavol=0.68, lweight=0.26, age=-0.14, lbph=0.21, svi=0.31, lcp=-0.29, gleason=-0.02, pgg45=0.27)
    df["lpsa"] = 2.47 + sum(c * zs[k] for k, c in coef.items()) + rng.normal(0, 0.71, n)
    for col in ("lcavol", "lweight", "lbph", "lcp", "lpsa"):
        df[col] = np.round(df[col], 5)
    df["train"] = rng.permutation(np.r_[np.ones(67, bool), np.zeros(30, bool)])
    return df[names + ["lpsa", "train"]]


def simulate_genotypes(rng: np.random.Generator, n_samples: int, n_snps: int, n_pops: int = 1,
                       fst: float = 0.0, maf_range=(0.05, 0.5)):
    """Simulate 0/1/2 genotypes under Hardy–Weinberg equilibrium.

    Each SNP gets an ancestral allele frequency drawn uniformly from ``maf_range``.  With
    ``n_pops > 1`` and ``fst > 0`` every population's frequency is drawn from the
    Balding–Nichols model Beta(p(1−F)/F, (1−p)(1−F)/F).  Samples are split evenly across
    populations.  Returns ``(G, pop, freqs)``: the N × J genotype matrix, the population of
    each sample and the n_pops × J matrix of population allele frequencies.

    ``G`` is stored as int8 to save memory: convert with ``G.astype(float)`` before arithmetic
    whose intermediate results can exceed 127.  A rare SNP can come out monomorphic (no
    variation) in a small sample — skip such columns in per-SNP tests.
    """
    p = rng.uniform(*maf_range, n_snps)
    if n_pops > 1 and fst > 0:
        a, b = p * (1 - fst) / fst, (1 - p) * (1 - fst) / fst
        freqs = np.clip(rng.beta(a, b, size=(n_pops, n_snps)), 0.001, 0.999)
    else:
        freqs = np.tile(p, (n_pops, 1))
    pop = np.repeat(np.arange(n_pops), int(np.ceil(n_samples / n_pops)))[:n_samples]
    G = rng.binomial(2, freqs[pop]).astype(np.int8)
    return G, pop, freqs


def simulate_quantitative_trait(rng: np.random.Generator, G: np.ndarray, n_causal: int = 5,
                                h2: float = 0.5, causal=None):
    """Simulate a quantitative phenotype y = G·β + ε with ``n_causal`` causal SNPs.

    The genetic part is scaled so that it explains a fraction ``h2`` (the heritability) of
    the phenotypic variance.  Returns ``(y, beta, causal_idx)``.
    """
    N, J = G.shape
    causal_idx = np.sort(rng.choice(J, n_causal, replace=False) if causal is None else np.asarray(causal))
    beta = np.zeros(J)
    beta[causal_idx] = rng.choice([-1, 1], len(causal_idx)) * rng.uniform(0.5, 1.0, len(causal_idx))
    g = G @ beta
    scale = np.sqrt(h2 / g.var()) if g.var() > 0 else 0.0
    beta *= scale
    y = G @ beta + rng.normal(0, np.sqrt(1 - h2), N)
    return y, beta, causal_idx


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


print("🧰 Toolkit loaded:", ", ".join(['simulate_prostate_clinical', 'simulate_genotypes', 'simulate_quantitative_trait', 'standardize']))

---
## 1 · GWAS in one picture

> 📍 **Where we are.** Modules 02–05 gave you probability (the Gaussian, Hardy–Weinberg), vectors and matrices, derivatives
> and the likelihood/MLE/MAP recipe. This module uses *all* of them for the first time on a real learning problem. We start
> with the biology: how geneticists find the loci behind a trait, what their results look like, and how the data are laid
> out.

**Family studies vs GWAS.** *How can we identify the genetic loci responsible for a phenotype?* The traditional answer is
the **family-based study**: to map Huntington's disease, recruit families with affected members, genotype everyone, draw
the pedigree and trace which stretch of chromosome is passed on together with the disease. This works for rare diseases
that run in families and are caused by one gene, but families are hard to recruit, and most common diseases (cancer,
diabetes, Alzheimer's, Parkinson's) do not follow a single-gene pattern. A **genome-wide association study (GWAS)**
instead collects **unrelated individuals** from the general population. They are much easier to recruit in large numbers
(it is easy to find thousands of affected people), and the design is effective for **common** diseases influenced by
many genes.

**What a GWAS result looks like: the Manhattan plot.** A GWAS tests every SNP for association with the trait and
plots $-\log_{10}p$ (Section 5 explains $p$-values; for now: taller = stronger evidence) against the SNP's position, one
chromosome after another in alternating colours. Most SNPs form a low, noisy band; SNPs near a locus that truly matters
rise above it like skyscrapers — hence the name. In the breast-cancer GWAS mentioned above (Turnbull *et al.* 2010;
582,886 SNPs, 3,659 cases, 4,897 controls) the tallest peak — the "high association" — sat on chromosome 10, in the
region of the gene *FGFR2*, far above the genome-wide significance line; a few other peaks also crossed the line, and most
SNPs stayed well below it. The next cell draws a Manhattan plot with this shape from simulated test
statistics — neighbouring SNPs are correlated, so each true signal appears as a *tower* of SNPs rather than a single point
(the reason, linkage disequilibrium, is explained in Section 5.5).

In [ ]:
# ── 1.1 A Manhattan plot shaped like a breast-cancer GWAS (SIMULATED test statistics) ──
rng_mh = np.random.default_rng(SEED + 1)
chrom_mb = np.array([249, 242, 198, 190, 181, 171, 159, 145, 138, 134, 135, 133,
                     114, 107, 102, 90, 83, 80, 59, 64, 47, 51])          # lengths of chromosomes 1–22 in Mb
n_snp_chr = np.round(60_000 * chrom_mb / chrom_mb.sum()).astype(int)    # 60,000 SNPs, spread by chromosome length
kernel = 0.9 ** np.arange(40)                                           # LD: neighbouring SNPs share information
peaks = {10: (0.92, 11.5), 16: (0.55, 8.0), 5: (0.30, 6.3), 2: (0.80, 6.0), 8: (0.60, 5.7)}  # chrom: (where, height)
z_all, chr_all = [], []
for c, n_c in enumerate(n_snp_chr, start=1):
    e = rng_mh.normal(size=n_c + len(kernel))
    z = np.convolve(e, kernel, mode="valid")[:n_c] / np.sqrt(np.sum(kernel ** 2))   # correlated N(0, 1) noise
    if c in peaks:                                                       # a causal locus: signal decays with distance
        where, height = peaks[c]
        z += height * np.exp(-np.abs(np.arange(n_c) - int(where * n_c)) / 12)
    z_all.append(z); chr_all.append(np.full(n_c, c))
z_all, chr_all = np.concatenate(z_all), np.concatenate(chr_all)
p_mh = 2 * stats.norm.sf(np.abs(z_all))                                 # two-sided p-value of each SNP

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.scatter(np.arange(len(p_mh)), -np.log10(p_mh), s=2, c=np.where(chr_all % 2, "tab:blue", "tab:green"), rasterized=True)
ax.axhline(-np.log10(5e-8), color="k", ls="--", lw=1, label="genome-wide significance, p = 5×10$^{-8}$")
mid = np.cumsum(n_snp_chr) - n_snp_chr / 2
ax.set_xticks(mid); ax.set_xticklabels([str(c) if c <= 12 or c % 2 == 0 else "" for c in range(1, 23)])
top = int(np.argmin(p_mh))
ax.annotate("high association", xy=(top, -np.log10(p_mh[top])), xytext=(top + 4000, -np.log10(p_mh[top]) - 2),
            color="tab:red", arrowprops=dict(arrowstyle="->", color="tab:red"), fontsize=8)
ax.set_xlabel("chromosome"); ax.set_ylabel("−log10 p"); ax.legend(fontsize=7, loc="upper left")
ax.set_title("Manhattan plot of a SIMULATED case–control GWAS (60,000 SNPs, five true loci)")
plt.tight_layout(); plt.show()
hits_mh = p_mh < 5e-8
print(f"{len(p_mh):,} SNPs; {hits_mh.sum()} pass p < 5e-8, on chromosomes {sorted(set(chr_all[hits_mh].tolist()))}; "
      f"strongest −log10 p = {-np.log10(p_mh.min()):.1f}; highest background SNP (outside the five loci): "
      f"{-np.log10(p_mh[~np.isin(chr_all, list(peaks))].min()):.1f}")

🔍 **What just happened?** We generated one test statistic per SNP: pure noise almost everywhere (correlated between
neighbours, as real SNPs are), plus a signal that fades with distance around five "true" loci. The result has the
anatomy of a published Manhattan plot: a dense band reaching up to $-\log_{10}p\approx6$ — the largest of 60,000 null
tests — and towers of neighbouring SNPs at the true loci, the tallest on chromosome 10 ($-\log_{10}p\approx30$). Only
SNPs above the dashed genome-wide line count as discoveries: 34 SNPs in four towers (chromosomes 2, 8, 10, 16). The
fifth true locus, on chromosome 5, stays just below the line in this sample — a real effect that a study of this size
misses (Section 5.4 calls this a lack of *power*). In the rest of the notebook we build the
machinery that produces such a plot, starting from a single SNP.

**Two kinds of phenotype.** The measurement of interest decides the kind of learning problem:

* *discrete* phenotypes — 0 = healthy, 1 = diabetic, as in a **case–control** study like the breast-cancer GWAS above —
  give a **classification** problem (Modules 08–11);
* *real-valued* phenotypes — the expression level of a gene such as *BRCA1* or *CFTR*, blood pressure, insulin level —
  give a **regression** problem, today's topic.

Regression is far more general than genetics: predict a person's weight from sex, height and age; today's price of a stock
from yesterday's prices of several companies; each pixel of a robot's next camera image from the current image and the
robot's action. Whenever the output is a continuous number and we have input–output pairs, regression applies; genetics
just keeps it concrete. In the language of Module 01, regression is **supervised learning**: we learn a function
$f: X\to Y$, or more precisely the conditional distribution $P(Y\mid X)$, from examples in which both $X$ and $Y$ are
observed. Learning one output from **one** input is *univariate* regression; from **many** inputs, *multivariate*
regression.

**Coding genotypes as numbers.** You carry two copies of each chromosome, one from each parent. At a SNP where the
population has the letters `A` and `T`, each person is `AA`, `AT` or `TT`. We pick one allele — usually the rarer one, the
*minor allele*, here `T` — and **count its copies**:

| genotype | AA | AT | TT |
|---|---|---|---|
| code $x$ | 0 | 1 | 2 |

A classic classroom cartoon of a GWAS shows seven short DNA sequences — four from people labelled *healthy*, three from
people with *cancer* — and asks: *which of the SNPs are responsible for determining the phenotype?* Let us find the SNPs
ourselves.

In [ ]:
# ── 1.2 Finding the SNPs in seven short DNA sequences ──
seqs = ["ACTCGTACGTAGACCTAGCATTACGCAATAATGCGA",       # healthy
        "ACTCGAACCTAGACCTAGCATTACGCAATAATGCGA",       # healthy
        "TCTCGTACGTAGACGTAGCATTACGCAATTATCCGA",       # healthy
        "ACTCGAACCTAGACCTAGCATTACGCAATTATCCGA",       # healthy
        "ACTCGTACGTAGACGTAGCATAACGCAATAATGCGA",       # cancer
        "TCTCGTACCTAGACGTAGCATAACGCAATAATCCGA",       # cancer
        "ACTCGAACCTAGACCTAGCATAACGCAATTATCCGA"]       # cancer
status = np.array([0, 0, 0, 0, 1, 1, 1])                   # 0 = healthy, 1 = cancer
letters = np.array([list(s) for s in seqs])                 # 7 people × 36 positions (a 2-D array of characters)
snp_positions = [j for j in range(letters.shape[1]) if len(set(letters[:, j])) > 1]
print(f"{letters.shape[1]} positions; {len(snp_positions)} of them vary between people (these are the SNPs)")

fig, ax = plt.subplots(figsize=(11, 2.4))
ax.axhspan(-0.5, 3.5, color="tab:orange", alpha=0.10); ax.axhspan(3.5, 6.5, color="tab:red", alpha=0.15)
for j in snp_positions:
    ax.axvspan(j - 0.45, j + 0.45, color="tab:green", alpha=0.35)
for i, s in enumerate(seqs):
    for j, ch in enumerate(s):
        ax.text(j, i, ch, ha="center", va="center", fontsize=8, family="monospace")
ax.set_xlim(-0.6, 35.6); ax.set_ylim(6.6, -0.6)
ax.set_yticks(range(7)); ax.set_yticklabels([f"healthy {i + 1}" for i in range(4)] + [f"cancer {i + 1}" for i in range(3)])
ax.set_xticks(snp_positions); ax.set_xticklabels([p + 1 for p in snp_positions], fontsize=7)
ax.set_xlabel("position (SNP positions highlighted in green)")
ax.set_title("a cancer GWAS in miniature: which SNPs are responsible for the phenotype?")
plt.tight_layout(); plt.show()

rows = []
for j in snp_positions:
    column = letters[:, j]
    alleles, counts = np.unique(column, return_counts=True)
    counted = alleles[np.argmin(counts)]                    # count the rarer letter, so x = 1 means "carries it"
    x_code = (column == counted).astype(int)
    rows.append({"position": j + 1, "letters (7 people)": "".join(column), "counted allele": counted,
                 "x: healthy | cancer": "".join(map(str, x_code[:4])) + " | " + "".join(map(str, x_code[4:])),
                 "freq. healthy": x_code[status == 0].mean(), "freq. cancer": x_code[status == 1].mean()})
pd.DataFrame(rows)

🔍 **What just happened?** Of the 36 positions, only 7 vary — these are the SNPs (green columns). At position 22 every
healthy person carries `T` and every cancer patient carries `A`: a perfect separation, and the obvious candidate answer to
the cartoon's question. The other SNPs are mixed.

⚠️ **Do not over-read this.** With 7 people, *some* position is likely to separate the groups by pure chance. The cartoon
is a cartoon: a real GWAS needs thousands of people and a statistical test that says how surprising a pattern is
(Section 5). Also note that each sequence here is *one* chromosome copy, so its codes are 0/1; a real diploid genotype
adds the two copies and gives 0/1/2.

✏️ **Your turn.** Change one letter: give the last cancer patient a `T` at position 22
(`seqs[6] = seqs[6][:21] + "T" + seqs[6][22:]`) and re-run the cell. What happens to the "perfect" SNP? How much
should a single person be able to change a scientific conclusion?

**The data of a GWAS.** Stack the phenotypes of $N$ individuals into a column vector $\mathbf y$ (the *phenotype data*)
and their genotypes into an $N\times J$ matrix $\mathbf X$ (the *genotype data*) — **one row per person, one column per
SNP**. We write a person's index as a *superscript* and a SNP's index as a *subscript*:

$$\mathbf y = \begin{pmatrix} y^1\\ \vdots\\ y^N\end{pmatrix},\qquad
  \mathbf X = \begin{pmatrix} x^1_1 & \cdots & x^1_J\\ \vdots & & \vdots\\ x^N_1 & \cdots & x^N_J\end{pmatrix},
  \qquad x^i_j\in\{0,1,2\}.$$

*In words:* $y^i$ is the trait value of person $i$; $x^i_j$ is how many copies of the counted allele person $i$ carries at
SNP $j$. (The superscript $i$ is a label, not a power.) In NumPy this is `y[i]` and `X[i, j]`. Row $i$ of $\mathbf X$
together with entry $i$ of $\mathbf y$ is one training example $\langle\mathbf x^i, y^i\rangle$.

⚠️ Some texts and software store the transpose (SNPs in rows, people in columns — common in genotype file formats). Always
check which way round a matrix is before you compute with it; in this course it is **row = individual $i$, column = SNP
$j$**, with $N$ individuals and $J$ SNPs.

---
## 2 · A simulated GWAS cohort

> 📍 **Where we are.** We know the layout ($\mathbf y$, $\mathbf X$). To judge a method we need data where we *know the
> answer*, so we simulate a cohort with the course toolkit (the same Hardy–Weinberg sampling you met in Module 02).

The simulation:

* $N = 500$ unrelated people, $J = 2{,}000$ SNPs. For every SNP an allele frequency $f_j$ is drawn between 0.05 and 0.5, and
  each person's genotype is a draw from Binomial$(2, f_j)$ — two independent allele copies (Hardy–Weinberg).
* **8 causal SNPs** get a non-zero effect $w_j$; the other 1,992 have $w_j = 0$.
* The phenotype is $y^i = \sum_j x^i_j w_j + \epsilon^i$, scaled so that the genotypes explain 40 % of its variance — the
  **heritability** $h^2 = 0.4$ — and the noise $\epsilon$ (environment, measurement error, all the genes we did not
  simulate) the remaining 60 %. Many complex human traits have heritabilities in the range 0.3–0.8. The phenotype is in
  standard-deviation units (its variance is about 1).

In [ ]:
# ── 2.1 Simulate a GWAS cohort: 500 people × 2,000 SNPs, 8 causal SNPs, heritability 0.4 ──
rng = np.random.default_rng(SEED + 2)        # re-seed: this section's results do not depend on earlier cells
N, J = 500, 2000
G, pop, freqs = simulate_genotypes(rng, N, J)                    # G[i, j] ∈ {0, 1, 2}
y, w_true, causal = simulate_quantitative_trait(rng, G, n_causal=8, h2=0.4)
f = freqs[0]                                                     # allele frequency of every SNP (one population)

print("genotype matrix X:", G.shape, " phenotype vector y:", y.shape)
print("how many 0s, 1s and 2s in X:", np.bincount(G.ravel()))
het_obs, het_hw = np.mean(G == 1), np.mean(2 * f * (1 - f))
print(f"fraction heterozygous: observed {het_obs:.3f}, Hardy–Weinberg expectation 2f(1−f) {het_hw:.3f}")
print("causal SNPs (the answer key — a real study never sees this):", causal)
print("their true effects w_j:", np.round(w_true[causal], 3))
pd.DataFrame(G[:5, :8], index=[f"person {i}" for i in range(5)],
             columns=[f"SNP {j}" for j in range(8)]).assign(phenotype_y=np.round(y[:5], 2))

In [ ]:
# ── 2.2 Look at the data before modelling it ──
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
im = axes[0].imshow(G[:40, :80], cmap="viridis", aspect="auto", interpolation="nearest")
axes[0].set_xlabel("SNP j (first 80)"); axes[0].set_ylabel("person i (first 40)")
axes[0].set_title("genotype matrix X (SIMULATED)")
cb = fig.colorbar(im, ax=axes[0], ticks=[0, 1, 2]); cb.ax.set_yticklabels(["AA = 0", "AT = 1", "TT = 2"])
axes[1].hist(G.mean(axis=0) / 2, bins=30, color="tab:blue", edgecolor="white")
axes[1].set_xlabel("allele frequency of T (mean genotype / 2)"); axes[1].set_ylabel("number of SNPs")
axes[1].set_title("2,000 SNPs: common variants (0.05–0.5)")
axes[2].hist(y, bins=30, color="tab:green", edgecolor="white")
axes[2].set_xlabel("phenotype y (SD units)"); axes[2].set_ylabel("number of people")
axes[2].set_title("a bell-shaped quantitative trait")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The genotype matrix is a noisy mosaic of 0/1/2 — no pattern is visible by eye. The observed
fraction of heterozygotes matches the Hardy–Weinberg prediction $2f(1-f)$ averaged over SNPs (Module 02). The phenotype is
bell-shaped, as expected for a trait built from several genetic effects plus Gaussian noise. Somewhere in those 2,000
columns hide 8 SNPs that matter; the rest of the notebook is about finding them.

✏️ **Your turn.** Change `h2=0.4` to `h2=0.1` and re-run Sections 2–5 later. Predict now: will the GWAS find more or fewer
causal SNPs?

---
## 3 · One SNP at a time: the univariate regression model

> 📍 **Where we are.** We first analyse **one SNP at a time** ("univariate regression"): take a single column of
> $\mathbf X$ and ask *does this SNP affect the phenotype?* Continuous traits such as cholesterol level or blood pressure
> are also called **quantitative traits**. The first step of any analysis is to **plot** the data — a scatter plot of
> phenotype against genotype, one point per person — and look for a trend before fitting anything.

In [ ]:
# ── 3.1 Scatter plots: a causal SNP and a null SNP ──
var_explained = 2 * f * (1 - f) * w_true ** 2          # variance a SNP adds to y: Var(x)·w² with Var(x) = 2f(1−f)
j_c = int(causal[np.argmax(var_explained[causal])])     # the causal SNP with the largest effect on the variance
j_0 = int(np.flatnonzero((w_true == 0) & (f > 0.3))[0]) # a null SNP with a similar, common allele frequency


def jitter(x, rng, width=0.08):
    """Shift 0/1/2 genotypes sideways by a small random amount so that overlapping points become visible."""
    return x + rng.uniform(-width, width, len(x))


fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
for ax, j, label in zip(axes, [j_c, j_0], ["causal", "null"]):
    x = G[:, j]
    ax.scatter(jitter(x, rng), y, s=6, alpha=0.35, color="tab:blue")
    means = [y[x == g].mean() for g in (0, 1, 2)]
    ax.plot([0, 1, 2], means, "k_", ms=30, mew=2.5, label="mean of each genotype group")
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["AA (0)", "AT (1)", "TT (2)"])
    ax.set_xlabel(f"genotype x at SNP {j}"); ax.set_title(f"SNP {j} ({label}, true w = {w_true[j]:+.2f})")
    print(f"SNP {j} ({label}): people per genotype {np.bincount(x, minlength=3)}, group means {np.round(means, 2)}")
axes[0].set_ylabel("phenotype y (SD units)"); axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** For the causal SNP the group means move steadily as the genotype goes from 0 to 2 copies — a
trend you can see despite the huge spread within each group. For the null SNP the means wiggle without a consistent trend.
Even the causal SNP explains only a small part of the variation: most of the spread comes from other genes and the
environment. Our job is to turn "I can see a trend" into numbers.

### 3.2 Step 1 of the recipe: choose a parameterised form for $P(Y\mid X;\theta)$

We wish to learn $f: X\to Y$, where $Y$ is real-valued, from $N$ training pairs
$\{\langle x^1,y^1\rangle,\dots,\langle x^N,y^N\rangle\}$. 02-620 uses the same **two-step statistical principle** for
every model in the course:

1. **Step 1:** choose a parameterised form for $P(Y\mid X;\theta)$ ($\theta$ = the vector of parameters);
2. **Step 2:** derive the learning algorithm as the **MLE** or **MAP** estimate of $\theta$ (Module 05).

Breaking learning into these two steps makes it manageable: once the probability model is set up, "learning" becomes the
concrete problem of estimating its parameters. We model $P(Y\mid X)$, not $P(X, Y)$, because this is **supervised
learning**: data are given for both $X$ and $Y$, the genotypes are taken as given, and we only want to explain the
phenotype given the genotype. ($Y$ is continuous here; $X$ may be continuous or discrete — our 0/1/2 genotypes are discrete.)

**Separate the certain from the uncertain.** The trend is the part we can capture; the scatter around it is the
part we cannot predict. So write the phenotype as a deterministic function plus random noise:

$$y = f(x) + \epsilon, \qquad \epsilon \sim \mathcal N(0, \sigma^2) .$$

*In words:* a person's phenotype is "what their genotype predicts" plus a random deviation that is, on average, zero and has
spread $\sigma$. The noise collects everything not in the model — other genes, diet, measurement error.

Adding a constant $f(x)$ to a Gaussian shifts its mean, so $y$ itself is Gaussian around $f(x)$:

$$p(y\mid x) = \mathcal N\big(f(x), \sigma^2\big).$$

*In words:* for each genotype there is a bell curve of possible phenotypes, centred on $f(x)$, all with the same width.

**Choose a linear $f$.** The simplest trend is a straight line, $f(x) = w_0 + w x$, which gives the model

$$\boxed{\,y = w_0 + w x + \epsilon,\qquad p(y\mid x; w_0, w, \sigma^2) = \mathcal N(w_0 + w x,\ \sigma^2)\,}$$

*In words:* every extra copy of the counted allele shifts the average phenotype by $w$; $w_0$ is the average phenotype of
`AA` people. This is called a **linear model** because it is linear in the parameters $w_0, w$.

🧮 **Math refresher — the Gaussian density (Module 02).** $\mathcal N(\mu,\sigma^2)$ has density
$$p(y) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\Big(-\frac{(y-\mu)^2}{2\sigma^2}\Big).$$
*In words:* the further $y$ is from the centre $\mu$, measured in units of $\sigma$, the (exponentially) less likely it is;
the factor in front makes the total area 1.

**What the sign of $w$ means.** $w = 0$: the SNP has no effect. $w > 0$: each copy of `T` raises the phenotype.
$w < 0$: each copy lowers it.

**Terminology** — you will meet all these names in papers:

| symbol | names |
|---|---|
| $y$ | dependent variable, output, response, phenotype |
| $x$ | explanatory variable, predictor, covariate, input |
| $w_0$ | intercept |
| $w$ | regression coefficient (slope, effect size) |
| $\epsilon$ | noise (error term) |

⚠️ In real data the spread of the phenotype often looks different for different genotypes — say larger for $x=0$ than
for $x=2$ — yet this model uses **one** $\sigma^2$ for all genotypes ("homoscedastic" noise: the bell curves all have the
same width). It is the simplest regression model; models whose noise level depends on $x$ exist but are not covered here.

In [ ]:
# ── 3.2 The model is a recipe for generating data ──
x_c = G[:, j_c].astype(float)
w0_guess = y[x_c == 0].mean()                            # a first, by-eye guess: mean of the AA group …
w_guess = (y[x_c == 2].mean() - y[x_c == 0].mean()) / 2  # … and half the TT–AA difference per allele copy
sigma_guess = np.std(y - (w0_guess + w_guess * x_c))
print(f"guessed parameters: w0 = {w0_guess:.3f}, w = {w_guess:.3f}, sigma = {sigma_guess:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4), sharey=True)
grid_y = np.linspace(y.min() - 0.5, y.max() + 0.5, 200)
axes[0].scatter(jitter(x_c, rng), y, s=5, alpha=0.25, color="tab:blue")
for g in (0, 1, 2):                                      # draw p(y | x = g) sideways at each genotype
    dens = stats.norm.pdf(grid_y, w0_guess + w_guess * g, sigma_guess)
    axes[0].plot(g + 0.9 * dens / dens.max() * 0.45, grid_y, color="tab:orange")
axes[0].plot([0, 2], [w0_guess, w0_guess + 2 * w_guess], "k-", lw=1.5, label="f(x) = w0 + w·x")
axes[0].set_title("the cohort, with p(y | x) = N(w0 + w·x, σ²)"); axes[0].legend(fontsize=7, loc="upper left")
y_fake = w0_guess + w_guess * x_c + rng.normal(0, sigma_guess, N)  # simulate NEW phenotypes from the model
axes[1].scatter(jitter(x_c, rng), y_fake, s=5, alpha=0.25, color="tab:purple")
axes[1].set_title("phenotypes simulated FROM the model")
for ax in axes:
    ax.set_xticks([0, 1, 2]); ax.set_xlabel(f"genotype x at SNP {j_c}")
axes[0].set_ylabel("phenotype y (SD units)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The orange curves are the model's three bell curves $p(y\mid x)$ for $x = 0, 1, 2$, centred on
$f(0)$, $f(1)$ and $f(2)$ — the standard picture of what a regression model assumes. On the right we *generated* phenotypes from the model: they look like the actual cohort on the left. That is what a
probability model is: a recipe that could have produced the data. Learning means picking the recipe's parameters so
that the observed data become as probable as possible — our by-eye guess is probably close, but there is no reason it
should be exactly the best.

✏️ **Your turn.** Set `w_guess = 0` and re-run the cell. What do the three bell curves look like now, and what would that say
about the SNP? Then try a negative value.

### 📘 Lecture notes — the regression model for genotype → phenotype

- **Data.** $N$ unrelated individuals, $J$ SNPs. Phenotypes in a vector $\mathbf y$ ($N\times1$), genotypes in a matrix
  $\mathbf X$ ($N\times J$, row = person, column = SNP), $x^i_j\in\{0,1,2\}$ = copies of the counted allele
  (AA = 0, AT = 1, TT = 2). A discrete phenotype (case/control) gives a classification problem, a real-valued one
  (gene expression, blood pressure, insulin level) a regression problem.
- **The two-step principle.** *Step 1:* choose a parameterised form for $P(Y\mid X;\theta)$. *Step 2:* derive the
  learning algorithm as the MLE or MAP estimate of $\theta$. Regression is supervised learning: we model $P(Y\mid X)$.
- **The univariate model** (one SNP at a time). Separate what we can capture (a trend $f$) from what we cannot (noise):
  $$y = f(x) + \epsilon,\quad \epsilon\sim\mathcal N(0,\sigma^2)\qquad\Longrightarrow\qquad p(y\mid x) = \mathcal N\big(f(x),\sigma^2\big).$$
  *In words:* each genotype has a bell curve of possible phenotypes centred on $f(x)$, all with the same width $\sigma$.
  With the linear choice $f(x) = w_0 + wx$:
  $$p(y\mid x; w_0, w, \sigma^2) = \mathcal N(w_0 + wx,\ \sigma^2).$$
  *In words:* each extra copy of the allele shifts the mean phenotype by $w$; $w_0$ is the mean phenotype of AA people.
- **Reading $w$.** $w = 0$: the SNP has no effect; $w > 0$: a positive effect; $w < 0$: a negative effect.
- **Terminology.** $y$: dependent variable, output, response. $x$: explanatory variable, predictor, covariate, input.
  $w_0$: intercept. $w$: regression coefficient. $\epsilon$: noise.
- **Common confusions.** "Linear" means linear in the *parameters* ($w_0 + w_1x + w_2x^2$ is still a linear model).
  "Noise" means "not explained by the model" (other genes, environment, measurement error), not "random in nature".
  The model assumes one $\sigma^2$ for every genotype, even if real data look more spread out for some genotypes.

> 🎓 **Exam tip.** Be able to write $p(y\mid x) = \mathcal N(w_0 + wx, \sigma^2)$ from $y = w_0 + wx + \epsilon$ and say which part is deterministic and which is random, and to state the two steps (model, then MLE/MAP) for any method of the course.

---
## 4 · Step 2: learning $w_0$ and $w$ — maximum likelihood is least squares

> 📍 **Where we are.** Step 1 gave us the model $p(y\mid x; w_0, w, \sigma^2) = \mathcal N(w_0 + wx, \sigma^2)$. Step 2
> asks: given $N$ training pairs $\{\langle x^1,y^1\rangle,\dots,\langle x^N,y^N\rangle\}$, which $w_0, w$
> should we pick? Module 05 answered this for a coin and a Gaussian mean: **maximise the likelihood**.

### 4.1 Why it is a *conditional* likelihood

The likelihood is the probability of the whole data set $D$ given the parameters. If the $N$ people are independent
(i.i.d.), it is a product over people, and each factor splits by the product rule $P(a, b) = P(a\mid b)P(b)$:

$$P(D\mid\theta) = \prod_{i=1}^{N} P(x^i, y^i\mid\theta) = \prod_{i=1}^{N} P(y^i\mid x^i,\theta)\,P(x^i\mid\theta).$$

*In words:* the chance of seeing everyone's data is the product of each person's chance; each person's chance is "the
chance of their genotype" times "the chance of their phenotype given their genotype".

Our regression parameters $w_0, w$ appear only in $P(y^i\mid x^i,\theta)$ — the model says nothing about how genotypes
arise — so the factors $P(x^i)$ are constants that cannot change *where* the maximum is. We maximise only

$$\underset{\theta}{\arg\max}\ \prod_{i=1}^{N} p\big(y^i\mid x^i, \theta\big),$$

*In words:* choose the parameters that make the observed phenotypes as probable as possible, *given* the observed genotypes.

This is the **maximum conditional likelihood estimate** (MCLE; you will also see the
abbreviation $\mathbf w_{MLCE}$ — same thing). People often just say "MLE", even for conditional models.

🧮 **Math refresher — products, sums, logs and arg max.**
* $\prod_{i=1}^N a_i = a_1 a_2\cdots a_N$ (multiply), $\sum_{i=1}^N a_i = a_1 + \dots + a_N$ (add).
* $\log(ab) = \log a + \log b$, so $\log\prod_i a_i = \sum_i \log a_i$; and $\log e^{z} = z$. (`log` is the natural log.)
* $\arg\max_\theta g(\theta)$ is *the value of $\theta$* at which $g$ is largest (not the largest value itself).
* Because $\log$ is strictly increasing, $\arg\max g = \arg\max \log g$; in particular
  $\arg\max\prod_i f_i = \arg\max\log\prod_i f_i = \arg\max\sum_i\log f_i$. Likewise adding a constant,
  or multiplying by a **positive** constant, does not move the arg max; multiplying by $-1$ turns an arg max into an arg min.

### 4.2 The derivation, one line at a time

Start from the conditional likelihood and apply only the rules above:

$$\begin{aligned}
\mathbf w_{MCLE} &= \underset{w_0,\,w}{\arg\max}\ \prod_{i=1}^{N} P\big(y^i\mid x^i, w_0, w, \sigma^2\big) &(1)\\
&= \underset{w_0,\,w}{\arg\max}\ \sum_{i=1}^{N} \log P\big(y^i\mid x^i, w_0, w, \sigma^2\big) &(2)\\
&= \underset{w_0,\,w}{\arg\max}\ \sum_{i=1}^{N}\Big[\log\frac{1}{\sqrt{2\pi\sigma^2}} - \frac12\cdot\frac{\big(y^i-(w_0+wx^i)\big)^2}{\sigma^2}\Big] &(3)\\
&= \underset{w_0,\,w}{\arg\max}\ \sum_{i=1}^{N} -\frac12\big(y^i-(w_0+wx^i)\big)^2 &(4)\\
&= \underset{w_0,\,w}{\arg\min}\ \sum_{i=1}^{N} \big(y^i-(w_0+wx^i)\big)^2 &(5)\\
&= \underset{w_0,\,w}{\arg\min}\ \sum_{i=1}^{N} \big(y^i-\hat y^i\big)^2,\qquad \hat y^i = w_0 + w x^i . &(6)
\end{aligned}$$

*In words, line by line:*
(1) the definition; (2) take the log — the maximiser does not move, and the product becomes a sum;
(3) plug in the Gaussian density: the log of $\frac{1}{\sqrt{2\pi\sigma^2}}e^{-z}$ is $\log\frac{1}{\sqrt{2\pi\sigma^2}} - z$;
(4) the first term does not contain $w_0$ or $w$ (it is the same constant for every candidate line), so drop it; and
$\frac{1}{\sigma^2}$ is a positive constant, so drop it too; (5) remove the minus sign by switching arg max to arg min;
(6) name the model's prediction $\hat y^i$ ("y-hat").

$$\boxed{\ \text{maximum conditional likelihood (Gaussian noise)} \;=\; \text{least squared error}\ }$$

*In words:* the best line is the one whose predictions are, in total, closest to the observed phenotypes, where "close" is
measured by the **squared** vertical distance. Least squares was used for two centuries before machine learning; the
derivation shows it is not an arbitrary choice but exactly what the Gaussian noise model implies:
the maximum conditional likelihood estimate is also called the **least-squared-error estimate**, and MLE provides a
probabilistic interpretation of least squares. Let us check
every step numerically.

In [ ]:
# ── 4.1 Check: log-likelihood = constant − SSE / (2σ²) on a whole grid of candidate lines ──
def log_likelihood(w0, w, sigma2, x, y):
    """Conditional log-likelihood Σ_i log N(y_i | w0 + w·x_i, σ²) of the univariate model (equation 3)."""
    resid = y - (w0 + w * x)
    return np.sum(np.log(1 / np.sqrt(2 * np.pi * sigma2)) - 0.5 * resid ** 2 / sigma2)


def sse(w0, w, x, y):
    """Sum of squared errors Σ_i (y_i − (w0 + w·x_i))² (equation 5)."""
    return np.sum((y - (w0 + w * x)) ** 2)


sigma2 = sigma_guess ** 2                                    # any fixed positive value works (see ✏️ below)
w0_grid = np.linspace(w0_guess - 0.6, w0_guess + 0.6, 121)
w_grid = np.linspace(w_guess - 0.6, w_guess + 0.6, 121)
LL = np.array([[log_likelihood(a, b, sigma2, x_c, y) for b in w_grid] for a in w0_grid])   # 14,641 lines
SSE = np.array([[sse(a, b, x_c, y) for b in w_grid] for a in w0_grid])
const = N * np.log(1 / np.sqrt(2 * np.pi * sigma2))
print("max |LL − (const − SSE/(2σ²))| over the grid:", np.abs(LL - (const - SSE / (2 * sigma2))).max())
i_ll, i_sse = np.unravel_index(LL.argmax(), LL.shape), np.unravel_index(SSE.argmin(), SSE.shape)
print("grid point with the largest log-likelihood: w0 = %.3f, w = %.3f" % (w0_grid[i_ll[0]], w_grid[i_ll[1]]))
print("grid point with the smallest squared error: w0 = %.3f, w = %.3f" % (w0_grid[i_sse[0]], w_grid[i_sse[1]]))

fig, ax = plt.subplots(figsize=(5, 3.8))
cs = ax.contour(w_grid, w0_grid, SSE, levels=15, cmap="viridis")
ax.clabel(cs, fontsize=6, fmt="%.0f")
ax.plot(w_grid[i_sse[1]], w0_grid[i_sse[0]], "r*", ms=12, label="minimum of SSE = maximum of LL")
ax.plot(w_guess, w0_guess, "ko", ms=5, label="our by-eye guess")
ax.set_xlabel("slope w"); ax.set_ylabel("intercept w0"); ax.legend(fontsize=7)
ax.set_title(f"sum of squared errors for SNP {j_c}: a bowl")
plt.tight_layout(); plt.show()

🔍 **What just happened?** At all 14,641 grid points the log-likelihood equals a constant minus $\text{SSE}/(2\sigma^2)$ up
to floating-point round-off (about $10^{-13}$), so the two surfaces are mirror images: the highest point of one is the
lowest point of the other, and both searches return the same grid point (very close to our by-eye guess, the black
dot). The SSE surface is a smooth **bowl** (its contours
are ellipses) with a single lowest point. A bowl has zero slope only at its bottom, so we can find the bottom with
calculus instead of a grid.

✏️ **Your turn.** Change `sigma2` to `0.1` or `10` and re-run. Does the best grid point move? Why not? (Look at line (4) of
the derivation.)

### 4.3 The closed-form solution

Write the squared-error **loss** as $L(w_0, w) = \sum_{i=1}^{N}\big(y^i - w_0 - w x^i\big)^2$. At the bottom of the bowl
both partial derivatives (Module 04) are zero. Using the chain rule, $\frac{\partial}{\partial w_0}(\dots)^2 =
2(\dots)\cdot(-1)$ and $\frac{\partial}{\partial w}(\dots)^2 = 2(\dots)\cdot(-x^i)$:

$$\frac{\partial L}{\partial w_0} = -2\sum_{i=1}^N \big(y^i - w_0 - w x^i\big) = 0,
\qquad
\frac{\partial L}{\partial w} = -2\sum_{i=1}^N x^i\big(y^i - w_0 - w x^i\big) = 0 .$$

*In words:* at the best line, the residuals $y^i - \hat y^i$ add up to zero, and so do the residuals weighted by genotype.

**Solve the first equation for $w_0$.** Divide by $-2N$: $\bar y - w_0 - w\bar x = 0$, where $\bar y = \frac1N\sum_i y^i$
and $\bar x = \frac1N\sum_i x^i$ are the means. So

$$w_0 = \bar y - w\,\bar x .$$

*In words:* the best line always passes through the point of means $(\bar x, \bar y)$.

**Substitute into the second equation.** Replacing $w_0$ gives $\sum_i x^i\big[(y^i-\bar y) - w(x^i-\bar x)\big] = 0$, i.e.
$\sum_i x^i(y^i-\bar y) = w\sum_i x^i(x^i-\bar x)$. Because $\sum_i (y^i - \bar y) = 0$ and $\sum_i (x^i - \bar x) = 0$, we may
replace the leading $x^i$ by $(x^i - \bar x)$ on both sides without changing either sum (we subtract $\bar x\cdot 0$). Hence

$$\boxed{\ w = \frac{\sum_{i=1}^N (x^i-\bar x)(y^i-\bar y)}{\sum_{i=1}^N (x^i-\bar x)^2} = \frac{\operatorname{cov}(x,y)}{\operatorname{var}(x)},
\qquad w_0 = \bar y - w\bar x\ }$$

*In words:* the slope is how much genotype and phenotype vary *together*, divided by how much the genotype varies on its
own. It is a special case of the multivariate regression formula — we will confirm that in Section 6.

In [ ]:
# ── 4.2 Closed-form least squares for one SNP, checked four ways ──
def fit_univariate(x, y):
    """Least-squares (= maximum conditional likelihood) intercept w0 and slope w for a single feature x."""
    x_bar, y_bar = x.mean(), y.mean()
    w = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar) ** 2)
    w0 = y_bar - w * x_bar
    return w0, w


w0_hat, w_hat = fit_univariate(x_c, y)
slope_np, intercept_np = np.polyfit(x_c, y, deg=1)          # NumPy's polynomial fit returns [slope, intercept]
lr = stats.linregress(x_c, y)                                # SciPy's simple linear regression
print(f"closed form   : w0 = {w0_hat:.4f}, w = {w_hat:.4f}")
print(f"best grid pt  : w0 = {w0_grid[i_sse[0]]:.4f}, w = {w_grid[i_sse[1]]:.4f}  (grid spacing 0.01)")
print(f"np.polyfit    : w0 = {intercept_np:.4f}, w = {slope_np:.4f}")
print(f"scipy linregress: w0 = {lr.intercept:.4f}, w = {lr.slope:.4f}")
print(f"true effect of this SNP in the simulation: w = {w_true[j_c]:.4f}")
resid = y - (w0_hat + w_hat * x_c)
print(f"sum of residuals = {resid.sum():.2e}, sum of genotype-weighted residuals = {(x_c * resid).sum():.2e}")

🔍 **What just happened?** The formula, the grid search, NumPy and SciPy agree (the grid is only accurate to its spacing of
0.01). The estimate (0.37) is in the right range but not equal to the true effect used in the simulation (0.42): a
different sample of 500 people would give a slightly different $\hat w$. The two "normal equations" of Section 4.3 hold: both residual sums are zero
up to round-off.

### 4.4 The noise variance and how much the SNP explains

The same recipe estimates $\sigma^2$: set $\partial\log L/\partial\sigma^2 = 0$ in line (3) — exactly the Gaussian-variance
MLE of Module 05 — and you get

$$\hat\sigma^2 = \frac1N\sum_{i=1}^N\big(y^i - \hat y^i\big)^2 .$$

*In words:* the estimated noise variance is the average squared residual. (Statistical tests use the slightly larger
$\frac{1}{N-2}\sum_i(y^i-\hat y^i)^2$, which corrects for the two fitted parameters — the same $N$ vs $N-1$ issue as in
Module 05.)

The fraction of the phenotype's variance explained by the line is $R^2 = 1 - \hat\sigma^2/\operatorname{var}(y)$.

In [ ]:
# ── 4.3 Noise variance and variance explained by one SNP ──
sigma2_hat = np.mean(resid ** 2)
R2 = 1 - sigma2_hat / np.var(y)
print(f"var(y) = {np.var(y):.3f},  sigma²_hat (MLE) = {sigma2_hat:.3f},  R² = {R2:.3f}")
print(f"variance this SNP adds in the simulation, 2f(1−f)w² = {var_explained[j_c]:.3f} "
      f"= {var_explained[j_c] / np.var(y):.3f} of var(y)")

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.3))
axes[0].scatter(jitter(x_c, rng), y, s=5, alpha=0.3, color="tab:blue")
xx = np.array([-0.2, 2.2])
axes[0].plot(xx, w0_hat + w_hat * xx, "tab:red", lw=2, label=f"fit: y = {w0_hat:.2f} + {w_hat:.2f}·x")
axes[0].set_xticks([0, 1, 2]); axes[0].set_xlabel(f"genotype x at SNP {j_c}"); axes[0].set_ylabel("phenotype y")
axes[0].legend(fontsize=7); axes[0].set_title("least-squares line")
axes[1].hist(resid, bins=30, density=True, color="lightgray", edgecolor="gray", label="residuals y − ŷ")
gg = np.linspace(resid.min(), resid.max(), 200)
axes[1].plot(gg, stats.norm.pdf(gg, 0, np.sqrt(sigma2_hat)), "tab:red", label="N(0, σ²_hat)")
axes[1].set_xlabel("residual (SD units)"); axes[1].set_ylabel("density"); axes[1].legend(fontsize=7)
axes[1].set_title("residuals look Gaussian, as the model assumes")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The strongest causal SNP explains only about 7 % of the phenotype's variance ($R^2 = 0.068$).
In the simulation it contributes $2f(1-f)w^2 = 0.090$, about 9 % of $\operatorname{var}(y)$ — the same range; the sample
shows a little less because our estimate $\hat w = 0.37$ came out below the true 0.42. The residuals — the "noise" from this SNP's
point of view — follow the assumed Gaussian well. Note that this noise includes the other seven causal SNPs: *noise* means
"not explained by the model", not "random in nature".

✏️ **Your turn.** (a) Fit the **null** SNP `j_0` with `fit_univariate`. Is $\hat w$ exactly zero? (b) Fit SNP `j_c` using only
the first 50 people (`x_c[:50], y[:50]`). How much does $\hat w$ change? Both answers motivate the next section: estimates
fluctuate, and we need to know by how much.

⚠️ **Least squares is sensitive to outliers.** Squaring makes one point that is 5 SD away count as much as 25 points that are
1 SD away. This follows from the Gaussian assumption (Gaussian tails are thin). If your trait has gross outliers, a
heavier-tailed noise model leads to a different criterion — a Laplace noise model gives the *absolute* error, whose
one-number version is the median (Module 04).

### 📘 Lecture notes — learning the parameters: maximum conditional likelihood = least squares

- **Question:** how do we learn $w_0, w$ from $N$ training pairs $\langle x^i, y^i\rangle$? **Answer:** maximise the
  likelihood of the data. With i.i.d. samples, $P(D\mid\theta) = \prod_i P(y^i\mid x^i,\theta)P(x^i\mid\theta)$, and only
  the first factor contains the regression parameters, so we maximise the **conditional** likelihood
  $$\mathbf w_{MCLE} = \underset{\theta}{\arg\max}\ \prod_{i=1}^N p(y^i\mid x^i,\theta).$$
  *In words:* make the observed phenotypes as probable as possible given the observed genotypes.
- **The six-line derivation.** $\arg\max\prod_i P(y^i\mid x^i,w_0,w,\sigma^2)$
  $\;=\;\arg\max\sum_i\log P(\cdot)$ (log is increasing: $\arg\max\prod_i f_i = \arg\max\log\prod_i f_i = \arg\max\sum_i\log f_i$)
  $\;=\;\arg\max\sum_i\big[\log\frac{1}{\sqrt{2\pi\sigma^2}} - \frac{(y^i-(w_0+wx^i))^2}{2\sigma^2}\big]$ (plug in the Gaussian)
  $\;=\;\arg\max\sum_i -\frac12(y^i-(w_0+wx^i))^2$ (drop constants)
  $\;=\;\arg\min\sum_i (y^i-(w_0+wx^i))^2$ (flip the sign)
  $\;=\;\arg\min\sum_i (y^i-\hat y^i)^2$ with $\hat y^i = w_0 + wx^i$.
  *In words:* under Gaussian noise, the most likely line is the one with the **least squared error** — the MCLE is
  also called the *least-squared-error estimate*, and MLE gives least squares a probabilistic interpretation: it is
  not an arbitrary criterion.
- **Closed form** (set both partial derivatives to zero):
  $$\hat w = \frac{\sum_i(x^i-\bar x)(y^i-\bar y)}{\sum_i(x^i-\bar x)^2},\qquad \hat w_0 = \bar y - \hat w\bar x .$$
  *In words:* slope = covariance of genotype and phenotype divided by the variance of the genotype; the line passes
  through the point of means. It is the special case $J = 1$ of the multivariate formula of Section 6.
- **Noise variance:** $\hat\sigma^2 = \frac1N\sum_i(y^i-\hat y^i)^2$, the mean squared residual; $\hat w$ itself does
  not depend on $\sigma^2$.
- **Sanity check from the lecture:** if a derivation ends in an expression you cannot explain in plain English, suspect
  an error — here the result ("fit the points as closely as possible") matches intuition.

> 🎓 **Exam tip.** The derivation is a classic exam question. The two moves students forget to justify: the $\log\frac{1}{\sqrt{2\pi\sigma^2}}$ term is dropped because it does not depend on $w_0, w$, and $1/\sigma^2$ is dropped because it is a **positive** constant. Also be able to say why $P(x^i\mid\theta)$ disappears.

---
## 5 · Scanning the genome: one regression per SNP

> 📍 **Where we are.** We can fit one SNP. A GWAS fits **every** SNP separately and asks which slopes are convincingly
> different from zero. Deciding "convincingly" needs two tools from statistics — the **standard error** and the
> **$p$-value** — because every GWAS paper (and every Manhattan plot, like the one in Section 1) is built from them.

**How uncertain is $\hat w$?** If we could repeat the study with 500 new people, $\hat w$ would come out slightly different
each time (you saw this in the ✏️ exercise). The typical size of that fluctuation is the **standard error**:

$$\operatorname{SE}(\hat w) = \sqrt{\frac{s^2}{\sum_{i=1}^N (x^i-\bar x)^2}},\qquad s^2 = \frac{1}{N-2}\sum_{i=1}^N (y^i-\hat y^i)^2 .$$

*In words:* the slope is uncertain when the noise $s^2$ is large and when the genotypes vary little (a rare SNP has few
carriers, so its slope rests on few people). More people make the denominator grow, so the SE shrinks roughly like
$1/\sqrt N$.

**Is the slope surprisingly far from zero?** Divide the estimate by its standard error:

$$t = \frac{\hat w}{\operatorname{SE}(\hat w)} .$$

*In words:* $t$ says how many standard errors the estimate lies away from zero.

🧮 **Math refresher — hypothesis tests and $p$-values in one paragraph.** Assume, for the sake of argument, the **null
hypothesis** $H_0$: "this SNP has no effect" ($w = 0$). Under $H_0$ and the Gaussian model, $t$ follows a known bell-shaped
distribution — Student's $t$ distribution with $N-2$ **degrees of freedom** (the number of people minus the two fitted
parameters $w_0, w$), practically a standard Gaussian when $N$ is in the hundreds. The **$p$-value** is the probability, *if $H_0$ were true*, of getting a $|t|$ at least as large as the one
observed. A tiny $p$ means "a slope this large would rarely appear by chance alone". A $p$-value is **not** the probability
that $H_0$ is true, and it says nothing about the size or biological importance of the effect. GWAS papers plot
$-\log_{10} p$ so that $p = 10^{-8}$ becomes a tall "8".

In [ ]:
# ── 5.1 A vectorised GWAS scan: slope, SE, t and p for all 2,000 SNPs at once ──
def gwas_scan(G, y):
    """Univariate least-squares regression of y on every column of G.

    Returns arrays (w, se, t, p): slope, standard error, t-statistic and two-sided p-value per SNP.
    SNPs that do not vary in the sample get NaN.
    """
    n = len(y)
    X = G.astype(float)
    xc = X - X.mean(axis=0)                   # centre every SNP (broadcasting subtracts each column's mean)
    yc = y - y.mean()
    sxx = np.sum(xc ** 2, axis=0)             # Σ (x − x̄)² for every SNP
    ok = sxx > 0
    w, se = np.full(X.shape[1], np.nan), np.full(X.shape[1], np.nan)
    w[ok] = (xc[:, ok].T @ yc) / sxx[ok]      # Section 4's slope formula, for all SNPs in one matrix product
    rss = np.sum(yc ** 2) - w[ok] ** 2 * sxx[ok]   # residual sum of squares (✏️ below: why?)
    se[ok] = np.sqrt(rss / (n - 2) / sxx[ok])
    t = w / se
    p = 2 * stats.t.sf(np.abs(t), df=n - 2)   # two-sided: |t| at least this large, in either direction
    return w, se, t, p


t0 = time.perf_counter()
w_scan, se_scan, t_scan, p_scan = gwas_scan(G, y)
print(f"scanned {J} SNPs in {time.perf_counter() - t0:.3f} s")
for j in [j_c, j_0, 7]:                       # compare with SciPy, SNP by SNP
    ref = stats.linregress(G[:, j], y)
    print(f"SNP {j:4d}: ours w={w_scan[j]:+.4f} SE={se_scan[j]:.4f} p={p_scan[j]:.3e} | "
          f"scipy w={ref.slope:+.4f} SE={ref.stderr:.4f} p={ref.pvalue:.3e}")

🔍 **What just happened?** One matrix product computed all 2,000 slopes in milliseconds, and the slopes, standard errors and
$p$-values agree with SciPy's `linregress`. ✏️ **Your turn (algebra):** show that
$\sum_i\big(y_c^i - \hat w x_c^i\big)^2 = \sum_i (y_c^i)^2 - \hat w^2\sum_i (x_c^i)^2$ for centred $x_c, y_c$, which is the
`rss` line. (Hint: expand the square and use $\hat w\sum_i (x^i_c)^2 = \sum_i x^i_c y^i_c$.)

Before trusting the $p$-values, let us check the claim "under $H_0$, $t$ follows a $t$ distribution" **by simulation**: shuffle
the phenotypes among people (which destroys any genotype–phenotype link, so $H_0$ is true by construction) and recompute
$t$ many times.

In [ ]:
# ── 5.2 Check the null distribution of t by shuffling phenotypes ──
x0 = G[:, j_0].astype(float)
xc0 = x0 - x0.mean()
Y_shuffled = np.array([rng.permutation(y) for _ in range(4000)])     # 4,000 shuffled phenotype vectors
Yc = Y_shuffled - Y_shuffled.mean(axis=1, keepdims=True)
w_null = Yc @ xc0 / np.sum(xc0 ** 2)
rss_null = np.sum(Yc ** 2, axis=1) - w_null ** 2 * np.sum(xc0 ** 2)
t_null = w_null / np.sqrt(rss_null / (N - 2) / np.sum(xc0 ** 2))
print(f"fraction of shuffles with |t| > 1.96: {np.mean(np.abs(t_null) > 1.96):.3f}  (theory: 0.05)")
print(f"fraction with p < 0.01: {np.mean(2 * stats.t.sf(np.abs(t_null), N - 2) < 0.01):.4f}  (theory: 0.01)")
print(f"largest |t| among the 4,000 shuffles: {np.abs(t_null).max():.2f};  causal SNP {j_c}: t = {t_scan[j_c]:.2f}")

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.hist(t_null, bins=50, density=True, color="lightgray", edgecolor="gray", label="t from 4,000 shuffles")
tt = np.linspace(-4.5, 4.5, 300)
ax.plot(tt, stats.t.pdf(tt, df=N - 2), "tab:red", label=f"Student t, {N - 2} d.f.")
ax.axvline(t_scan[j_c], color="tab:blue", ls="--", label=f"causal SNP {j_c}: t = {t_scan[j_c]:.1f}")
ax.set_xlabel("t statistic"); ax.set_ylabel("density"); ax.legend(fontsize=7)
ax.set_title("t under 'no effect' matches the theory")
plt.tight_layout(); plt.show()

🔍 **What just happened?** When there is truly no effect, $|t| > 1.96$ happens about 5 % of the time and $p < 0.01$ about
1 % of the time, as the theory says, and the histogram of shuffled $t$ values sits on the Student-$t$ curve. The causal SNP's
observed $t \approx 6$ (blue dashed line) lies far to the right of anything the 4,000 shuffles produced — which is exactly
what a tiny $p$-value expresses.

### 5.3 The Manhattan plot and the multiple-testing problem

With 2,000 tests at the 5 % level, about 100 null SNPs would be "significant" by chance. The simplest fix, the
**Bonferroni correction**, divides the significance level by the number of tests: declare a hit only if
$p < 0.05/J$. Then the probability of *even one* false positive among all $J$ null SNPs is at most 5 %. Human GWAS use the
conventional **genome-wide significance** threshold $p < 5\times10^{-8}$, i.e. $0.05$ divided by about one million
independent common-variant tests.

A **Manhattan plot** (Section 1) shows $-\log_{10} p$ for every SNP in genome order, chromosome by chromosome; true signals
stand out like skyscrapers. Our simulated SNPs have no real genomic positions, so we simply place them on 10 pretend
chromosomes of 200 SNPs.

In [ ]:
# ── 5.3 Manhattan plot with the Bonferroni threshold ──
bonferroni = 0.05 / J
hits = np.flatnonzero(p_scan < bonferroni)
chrom = np.arange(J) // 200 + 1                                   # 10 pretend chromosomes of 200 SNPs each

fig, ax = plt.subplots(figsize=(11, 3.4))
colors = np.where(chrom % 2 == 0, "tab:blue", "tab:green")
ax.scatter(np.arange(J), -np.log10(p_scan), s=6, c=colors)
ax.scatter(causal, -np.log10(p_scan[causal]), s=60, facecolors="none", edgecolors="tab:red", lw=1.5,
           label="true causal SNPs (answer key)")
ax.axhline(-np.log10(bonferroni), color="k", ls="--", lw=1, label=f"Bonferroni 0.05/{J} = {bonferroni:.1e}")
ax.axhline(-np.log10(0.05), color="0.6", ls=":", lw=1, label="p = 0.05 (no correction)")
ax.set_xticks(np.arange(10) * 200 + 100); ax.set_xticklabels([f"chr{c}" for c in range(1, 11)])
ax.set_xlabel("SNP position (pretend chromosomes)"); ax.set_ylabel("−log10 p")
ax.set_title("Manhattan plot of a SIMULATED GWAS (500 people, 2,000 SNPs, h² = 0.4)"); ax.legend(fontsize=7, loc="upper right")
plt.tight_layout(); plt.show()

table = pd.DataFrame({"SNP": hits, "w_hat": w_scan[hits].round(3), "SE": se_scan[hits].round(3),
                      "p": p_scan[hits], "truly causal?": np.isin(hits, causal)})
print(f"{len(hits)} SNPs pass Bonferroni; {np.isin(causal, hits).sum()} of the 8 causal SNPs are among them; "
      f"{(~np.isin(hits, causal)).sum()} false positives")
table

🔍 **What just happened?** The causal SNPs (red circles) tower above the background, and the hits that pass Bonferroni are
all truly causal — no false positives in this run. But **not all eight** causal SNPs pass: some sit only modestly above
the noise. Without correction, the grey dotted line ($p = 0.05$) would let through about a hundred null SNPs. Let us see
both effects explicitly: what the background looks like under pure noise, and why some causal SNPs are missed.

In [ ]:
# ── 5.4 Null scan (shuffled phenotype) and the power to detect each causal SNP ──
_, _, _, p_shuffled = gwas_scan(G, rng.permutation(y))
print(f"shuffled phenotype: {np.sum(p_shuffled < 0.05)} SNPs with p < 0.05 (expected ≈ {0.05 * J:.0f}), "
      f"{np.sum(p_shuffled < bonferroni)} pass Bonferroni, smallest p = {np.nanmin(p_shuffled):.1e}")

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.0), sharey=True)
axes[0].hist(p_shuffled, bins=20, color="0.6", edgecolor="white"); axes[0].set_title("p-values, shuffled phenotype (H0 true)")
axes[1].hist(p_scan, bins=20, color="tab:blue", edgecolor="white"); axes[1].set_title("p-values, actual phenotype")
for ax in axes:
    ax.set_xlabel("p-value")
axes[0].set_ylabel("number of SNPs")
plt.tight_layout(); plt.show()

power = pd.DataFrame({"causal SNP": causal, "allele freq f": f[causal].round(2), "true w": w_true[causal].round(3),
                      "variance explained 2f(1−f)w²": var_explained[causal].round(4),
                      "−log10 p": (-np.log10(p_scan[causal])).round(1), "found?": np.isin(causal, hits)})
power.sort_values("variance explained 2f(1−f)w²", ascending=False)

🔍 **What just happened?** Under the null, $p$-values are spread **uniformly** between 0 and 1 (left): about 5 % fall below
0.05 purely by chance, and none reaches the Bonferroni line. With the actual phenotype (right) the histogram is uniform
*plus* a spike near 0 — the true signals. The **power** of a study is the probability that it detects an effect that is
really there; the table shows *which* causal SNPs are missed: the two that are missed are
among the three that explain the least variance, $2f(1-f)w^2$ — small effects or less common alleles. The third, SNP
1318 (allele frequency 0.10), was found only because its estimate happened to come out larger than its true effect (0.48
vs 0.33): near the threshold, luck decides in both directions. Real GWAS face exactly this
problem: thousands of variants with tiny effects, each individually undetectable without enormous sample sizes — one
reason the heritability of many traits is not fully accounted for by GWAS hits.

### 5.5 ⚠️ A GWAS hit is an association, not a cause

A significant SNP is *associated* with the phenotype. Two common reasons why it may not *cause* anything:

1. **Linkage disequilibrium (LD).** Neighbouring SNPs are inherited together, so their genotypes are correlated. A SNP can
   be significant only because it travels with the causal variant nearby. That is why real Manhattan plots show *towers* of
   neighbouring SNPs rather than single points, as in the plot of Section 1 (the SNPs of our cohort are simulated
   independently, so Section 5.3 shows isolated points). We meet LD again in Section 6.
2. **Population structure (confounding).** Suppose the cohort mixes two populations that differ slightly in allele
   frequencies at many SNPs *and* in an environmental factor (diet, say) that changes the phenotype. Then every SNP whose
   frequency differs between the populations looks associated — with **no genetic effect at all**. Let us simulate it.

In [ ]:
# ── 5.5 Population structure creates false hits (no SNP has any effect here) ──
rng_ps = np.random.default_rng(SEED + 15)
G_ps, pop_ps, _ = simulate_genotypes(rng_ps, 500, 2000, n_pops=2, fst=0.05)   # two populations, F_ST = 0.05
y_ps = 0.5 * pop_ps + rng_ps.normal(0, 1, 500)        # population 1 lives in a different environment: +0.5 SD
_, _, _, p_ps = gwas_scan(G_ps, y_ps)

# Fix: compare people only WITHIN their population (centre y and every SNP inside each population)
G_within, y_within = G_ps.astype(float), y_ps.copy()
for g in (0, 1):
    m = pop_ps == g
    G_within[m] -= G_within[m].mean(axis=0)
    y_within[m] -= y_within[m].mean()
_, _, _, p_within = gwas_scan(G_within, y_within)
print(f"naive scan          : {np.sum(p_ps < bonferroni):3d} SNPs pass Bonferroni, {np.sum(p_ps < 0.05)} have p < 0.05")
print(f"within-population   : {np.sum(p_within < bonferroni):3d} SNPs pass Bonferroni, {np.sum(p_within < 0.05)} have p < 0.05")

fig, axes = plt.subplots(1, 2, figsize=(11, 2.9), sharey=True)
for ax, p, title in zip(axes, [p_ps, p_within], ["naive scan: false 'hits' everywhere", "within-population scan"]):
    ax.scatter(np.arange(2000), -np.log10(p), s=5, c=colors)
    ax.axhline(-np.log10(bonferroni), color="k", ls="--", lw=1)
    ax.set_xlabel("SNP"); ax.set_title(title + " (SIMULATED, no causal SNPs)")
axes[0].set_ylabel("−log10 p")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Not a single SNP affects this phenotype, yet the naive scan reports several genome-wide
"significant" hits and roughly four times the expected number of $p<0.05$ results: the SNPs merely tag *which population*
a person comes from, and the population differs in environment. Comparing people only within their own population
removes the false signal. Real GWAS do the same thing more generally by adding the top **principal components** of the
genotype matrix as extra predictors — you will see in Module 14 that genotypes alone reveal ancestry (genes mirror
geography). Adding extra predictors is exactly what the *multivariate* regression of the next section allows.

✏️ **Your turn.** In cell 2.1, change `N, J = 500, 2000` to `N, J = 250, 2000` and re-run cells 2.1–5.4. How many causal SNPs
pass Bonferroni now? Then try `N = 1000`. Relate what you see to the formula for $\operatorname{SE}(\hat w)$.

### 📘 Lecture notes — genome-wide association studies

- **Why GWAS.** Family-based studies trace a disease through pedigrees: families are hard to recruit, and the approach
  suits rare, single-gene diseases that run in families. A GWAS genotypes **unrelated individuals** — easy to recruit
  in large numbers — and is effective for **common** diseases (cancer, diabetes) influenced by many loci.
- **The question.** *Which SNPs are responsible for (associated with) the phenotype?* Even seven short sequences show
  the idea: find the positions that vary, then ask which variation tracks the phenotype — and beware chance patterns
  in small samples.
- **Types of phenotype.** Discrete (0/1, case–control) → classification; real-valued (expression level, blood
  pressure, insulin level) → regression. Regression applies to any continuous output (weight from height and age,
  stock prices, image pixels).
- **The scan.** Fit $y = w_0 + w x_j + \epsilon$ for every SNP $j$; test $H_0: w = 0$ with
  $$t = \frac{\hat w}{\operatorname{SE}(\hat w)},\qquad \operatorname{SE}(\hat w) = \sqrt{\frac{s^2}{\sum_i(x^i-\bar x)^2}} .$$
  *In words:* how many standard errors the slope lies from zero; the $p$-value is the chance of an even larger $|t|$ if
  the SNP had no effect.
- **Multiple testing.** With $J$ tests, Bonferroni declares a hit only if $p < 0.05/J$; human GWAS use the genome-wide
  threshold $5\times10^{-8}$.
- **The Manhattan plot.** $-\log_{10}p$ against genomic position, chromosomes in alternating colours; true loci rise as
  towers above a noisy band. Example: in a breast-cancer GWAS of 582,886 SNPs in 3,659 cases and 4,897 controls
  (2010) the tallest peak, on chromosome 10 (*FGFR2* region), rose far above the genome-wide line, a few other peaks
  crossed it, and most SNPs stayed well below.
- **Caveats.** A hit is an *association*: it may only tag a nearby causal variant (linkage disequilibrium) or
  population ancestry (confounding). Power depends on sample size, allele frequency and effect size, so small effects
  are missed.

> 🎓 **Exam tip.** Know why GWAS plot $-\log_{10}p$, what Bonferroni does ($p < \alpha/J$), why the conventional threshold is $5\times10^{-8}$, and that a $p$-value is computed *assuming* no effect — it is not the probability that the SNP is causal.

---
## 6 · All SNPs at once: multivariate regression

> 📍 **Where we are.** The scan of Section 5 looks at each SNP *alone*. But in non-monogenic diseases several
> mutations act together, and features can interact: the effect of one may depend on the others. Multivariate regression
> estimates the effect of each SNP on the phenotype **in the presence of all the other SNPs**. The rule of thumb: analyse
> all features together whenever you can, and fall back to one-at-a-time regression only when the data are too large.
> Here is a small example of why that matters.

In [ ]:
# ── 6.1 A puzzle: two SNPs in linkage disequilibrium, only one of them causal ──
rng_ld = np.random.default_rng(SEED + 11)
n_ld = 1000
snp_a = rng_ld.binomial(2, 0.3, n_ld)                             # SNP A: the causal variant
same = rng_ld.random(n_ld) < 0.85                                  # LD: SNP B copies A's genotype in ~85 % of people
snp_b = np.where(same, snp_a, rng_ld.binomial(2, 0.3, n_ld))     # SNP B: a nearby, NON-causal SNP
y_ld = 0.4 * snp_a + rng_ld.normal(0, 1, n_ld)                     # only A affects the phenotype (true w_A = 0.4, w_B = 0)
print(f"correlation between the genotypes of A and B: {np.corrcoef(snp_a, snp_b)[0, 1]:.2f}")
w_ab, se_ab, _, p_ab = gwas_scan(np.column_stack([snp_a, snp_b]), y_ld)    # one SNP at a time (Section 5)
for k, name in enumerate(["A (causal)", "B (passenger)"]):
    print(f"univariate fit, SNP {name:14s} alone: w = {w_ab[k]:+.3f}, SE = {se_ab[k]:.3f}, p = {p_ab[k]:.1e}")

🔍 **What just happened?** SNP B has **no effect**, yet on its own it shows a slope almost as large as A's and an extremely
small $p$-value — it simply travels with A. One-at-a-time regression cannot tell the passenger from the driver. To ask
"does B matter *once A is accounted for*?" we must put both SNPs into one model. We return to this puzzle once we can fit
such a model.

### 6.1 The multivariate model

The two-step recipe is unchanged; only $x$ becomes a vector $\mathbf x = (x_1,\dots,x_J)^\top$ of $J$ genotypes (bold
letters denote vectors), and $w$ a vector $\mathbf w = (w_1,\dots,w_J)^\top$ of the same length:

$$y = w_0 + \mathbf x^\top\mathbf w + \epsilon = w_0 + x_1w_1 + x_2w_2 + \dots + x_Jw_J + \epsilon,\qquad \epsilon\sim\mathcal N(0,\sigma^2).$$

*In words:* the phenotype is a baseline plus a weighted sum of all genotypes — each SNP contributes its own effect $w_j$ per
allele copy. Picture one person's genotypes along the genome as a row ($\mathbf x^\top$), and next to it a column
$\mathbf w$ of **association strengths**, one per locus: the row times the column gives that person's predicted trait
value (say 2.1). A locus with a large $|w_j|$ has a strong effect on the trait; a locus with $w_j\approx0$ hardly any.
This weighted sum is also what geneticists call a **polygenic score**.

🧮 **Math refresher — vectors and transposes (Module 03).** Vectors are columns. The transpose $\mathbf x^\top$ turns a
column into a row, and a row times a column is the **dot product**: $\mathbf x^\top\mathbf w = \sum_{j=1}^J x_jw_j$ — a single
number. Rules we will need: $(\mathbf A\mathbf B)^\top = \mathbf B^\top\mathbf A^\top$ (transpose a product → reverse the
order); $(\mathbf A - \mathbf B)^\top = \mathbf A^\top - \mathbf B^\top$; a $1\times1$ "matrix" (a number) equals its own
transpose; and $\mathbf v^\top\mathbf v = \sum_i v_i^2 = \lVert\mathbf v\rVert^2$, the squared length.

**Augmenting the data.** The intercept is a nuisance in formulas. Trick: put a 1 on top of $\mathbf x$ and $w_0$ on
top of $\mathbf w$,

$$\mathbf x = \begin{pmatrix}1\\x_1\\ \vdots\\ x_J\end{pmatrix},\quad
  \mathbf w = \begin{pmatrix}w_0\\w_1\\ \vdots\\ w_J\end{pmatrix}
  \quad\Rightarrow\quad y = \mathbf x^\top\mathbf w + \epsilon = 1\cdot w_0 + x_1w_1 + \dots + x_Jw_J + \epsilon .$$

*In words:* the intercept becomes the weight of a pretend feature that equals 1 for everyone. Stacking all $N$ people's
augmented rows gives the $N\times(J+1)$ matrix $\mathbf X$ whose first column is all ones (the *augmented input feature* corresponding to $w_0$), and the model for
everyone at once reads $\mathbf y = \mathbf X\mathbf w + \boldsymbol\epsilon$.

In [ ]:
# ── 6.2 The augmented data matrix and the matrix–vector product ──
X_ld = np.column_stack([np.ones(n_ld), snp_a, snp_b])            # N × (J+1): a column of ones, then the SNPs
print("augmented X (first 4 people):\n", X_ld[:4])
w_try = np.array([0.1, 0.4, 0.0])                                 # some candidate weights (w0, wA, wB)
pred_loop = np.array([X_ld[i] @ w_try for i in range(n_ld)])     # person by person: x^iᵀ w
pred_matrix = X_ld @ w_try                                        # everyone at once: X w
print("loop and matrix product agree:", np.allclose(pred_loop, pred_matrix))

### 6.2 Step 2 in matrix form: the normal equations

Exactly as in Section 4 — independent people, Gaussian noise, take logs, drop constants — the conditional MLE is the
least-squares solution, now with $\hat y^i = \mathbf x^{i\top}\mathbf w$:

$$\mathbf w_{MCLE} = \underset{\mathbf w}{\arg\max}\ \sum_{i=1}^N -\big(y^i - \mathbf x^{i\top}\mathbf w\big)^2
 = \underset{\mathbf w}{\arg\min}\ \sum_{i=1}^N \big(y^i - \mathbf x^{i\top}\mathbf w\big)^2
 = \underset{\mathbf w}{\arg\min}\ (\mathbf y - \mathbf X\mathbf w)^\top(\mathbf y - \mathbf X\mathbf w).$$

*In words:* the vector $\mathbf y - \mathbf X\mathbf w$ holds all $N$ residuals, and a vector's transpose times itself is
the sum of its squared entries — so the last form is the same sum of squared errors, written compactly. (To code the first form you would loop over people and add up; the matrix form does it in one
expression.)

⚠️ **A common slip.** It is easy to write the middle expression as $\arg\max\sum_i\big[-(y^i - \mathbf x^{i\top}\mathbf w)\big]^2$.
With the minus sign *inside* the square the expression is just the (positive) squared error, and maximising it would be
wrong; the minus sign belongs *outside*: $-\big(y^i - \mathbf x^{i\top}\mathbf w\big)^2$, as written above.

**Differentiate and set to zero (equations 2–8).** Call the loss $L(\mathbf w) = (\mathbf y - \mathbf X\mathbf w)^\top(\mathbf y - \mathbf X\mathbf w)$.

$$\begin{aligned}
L(\mathbf w) &= (\mathbf y^\top - \mathbf w^\top\mathbf X^\top)(\mathbf y - \mathbf X\mathbf w) &(2)\\
&= \mathbf y^\top\mathbf y - \mathbf y^\top\mathbf X\mathbf w - \mathbf w^\top\mathbf X^\top\mathbf y + \mathbf w^\top\mathbf X^\top\mathbf X\mathbf w &(3)\\
&= \mathbf y^\top\mathbf y - 2\,\mathbf w^\top\mathbf X^\top\mathbf y + \mathbf w^\top\mathbf X^\top\mathbf X\mathbf w &(4)
\end{aligned}$$

*In words:* (2) the transpose of a difference is the difference of transposes, and $(\mathbf X\mathbf w)^\top = \mathbf
w^\top\mathbf X^\top$ (reverse the order); (3) multiply out the brackets like $(a-b)(c-d)$ — the order of factors must be
kept, matrices do not commute; (4) $\mathbf y^\top\mathbf X\mathbf w$ is a single number, so it equals its own transpose
$\mathbf w^\top\mathbf X^\top\mathbf y$, and the two middle terms combine.

Now take the gradient with the two rules of Module 04 (equations 69 and 81 of the *Matrix Cookbook*, a free reference
of matrix identities):
$\frac{\partial}{\partial\mathbf w}\mathbf w^\top\mathbf a = \mathbf a$ and
$\frac{\partial}{\partial\mathbf w}\mathbf w^\top\mathbf A\mathbf w = (\mathbf A + \mathbf A^\top)\mathbf w$.
The first term $\mathbf y^\top\mathbf y$ does not contain $\mathbf w$, so its derivative is $\mathbf 0$.

$$\begin{aligned}
\frac{\partial L}{\partial\mathbf w} &= -2\,\mathbf X^\top\mathbf y + \big(\mathbf X^\top\mathbf X + (\mathbf X^\top\mathbf X)^\top\big)\mathbf w &(5)\\
&= -2\,\mathbf X^\top\mathbf y + 2\,\mathbf X^\top\mathbf X\mathbf w \;=\; \mathbf 0 &(6)\\
\mathbf X^\top\mathbf X\,\mathbf w &= \mathbf X^\top\mathbf y &(7)\\
\mathbf w &= (\mathbf X^\top\mathbf X)^{-1}\mathbf X^\top\mathbf y &(8)
\end{aligned}$$

*In words:* (5) apply the two rules with $\mathbf a = \mathbf X^\top\mathbf y$ and $\mathbf A = \mathbf X^\top\mathbf X$;
(6) $\mathbf X^\top\mathbf X$ is symmetric (its transpose is $\mathbf X^\top\mathbf X$ again), so the bracket is
$2\mathbf X^\top\mathbf X$; set the gradient to zero; (7) rearrange — these $J+1$ linear equations are called the
**normal equations**; (8) multiply both sides on the left by $(\mathbf X^\top\mathbf X)^{-1}$, *if it exists*.

$$\boxed{\ \mathbf w_{MCLE} = (\mathbf X^\top\mathbf X)^{-1}\mathbf X^\top\mathbf y\ }$$

*In words:* one formula gives the best weights for all SNPs at once — a **closed-form** solution, found on paper by setting the gradient to zero. We are lucky: for most models later in
the course (logistic regression in Module 09, for example) setting the gradient to zero gives no one-line solution, and
we must optimise iteratively. The zero
gradient is a minimum (not a maximum) because $L$ is a convex bowl: $\mathbf w^\top\mathbf X^\top\mathbf X\mathbf w =
\lVert\mathbf X\mathbf w\rVert^2 \ge 0$.

Equation (7) also has a lovely reading: $\mathbf X^\top(\mathbf y - \mathbf X\mathbf w) = \mathbf 0$ says that **the residuals
are orthogonal to (uncorrelated with) every column of $\mathbf X$** — once the fit is optimal, no feature can explain any of
what is left over. For $J=1$ these are exactly the two equations of Section 4.3.

In [ ]:
# ── 6.3 Numerical checks of the matrix derivation ──
def rss(w, X, y):
    """Residual sum of squares (y − Xw)ᵀ(y − Xw)."""
    r = y - X @ w
    return r @ r


def rss_gradient(w, X, y):
    """Gradient of the residual sum of squares, −2Xᵀy + 2XᵀXw (equation 6)."""
    return -2 * X.T @ y + 2 * X.T @ X @ w


w_rand = rng.normal(size=3)
loop_sum = sum((y_ld[i] - X_ld[i] @ w_rand) ** 2 for i in range(n_ld))
expanded = y_ld @ y_ld - 2 * w_rand @ X_ld.T @ y_ld + w_rand @ X_ld.T @ X_ld @ w_rand     # equation (4)
print(f"Σ_i (y_i − x_iᵀw)² = {loop_sum:.6f} | (y−Xw)ᵀ(y−Xw) = {rss(w_rand, X_ld, y_ld):.6f} | equation (4) = {expanded:.6f}")
eps = 1e-6                                                         # finite differences: nudge one weight at a time
fd = np.array([(rss(w_rand + eps * e, X_ld, y_ld) - rss(w_rand - eps * e, X_ld, y_ld)) / (2 * eps) for e in np.eye(3)])
print("finite-difference gradient:", np.round(fd, 3))
print("formula −2Xᵀy + 2XᵀXw     :", np.round(rss_gradient(w_rand, X_ld, y_ld), 3))

🔍 **What just happened?** The person-by-person sum, the compact matrix form and the expanded equation (4) give the same
number, and the gradient formula matches a brute-force finite-difference gradient. The algebra holds. Now solve the normal
equations.

⚠️ **In code, never form the inverse.** Equation (8) is for paper. Numerically we *solve* the linear system (7) with
`np.linalg.solve`, which is faster and more accurate than computing `np.linalg.inv(X.T @ X)` and multiplying (Module 03).

In [ ]:
# ── 6.4 Multivariate least squares: solve the normal equations, check, and resolve the LD puzzle ──
def fit_ols(X, y):
    """Ordinary least squares: solve the normal equations XᵀX w = Xᵀy (X must already contain the column of ones)."""
    return np.linalg.solve(X.T @ X, X.T @ y)


def ols_table(X, y, names):
    """OLS estimates with standard errors, t-statistics and p-values (σ² estimated with N − (J+1) d.f.)."""
    n, k = X.shape
    w = fit_ols(X, y)
    s2 = rss(w, X, y) / (n - k)
    se = np.sqrt(s2 * np.diag(np.linalg.inv(X.T @ X)))     # Var(ŵ) = σ²(XᵀX)⁻¹ — the inverse IS needed here
    t = w / se
    return pd.DataFrame({"coef": w, "SE": se, "t": t, "p": 2 * stats.t.sf(np.abs(t), n - k)}, index=names)


w_ols = fit_ols(X_ld, y_ld)
w_lstsq = np.linalg.lstsq(X_ld, y_ld, rcond=None)[0]                          # NumPy's least-squares solver
sk = LinearRegression().fit(np.column_stack([snp_a, snp_b]), y_ld)           # scikit-learn adds the intercept itself
print("normal equations :", np.round(w_ols, 4))
print("np.linalg.lstsq  :", np.round(w_lstsq, 4))
print("scikit-learn     :", np.round(np.r_[sk.intercept_, sk.coef_], 4))
print("Xᵀ(y − Xŵ) =", np.round(X_ld.T @ (y_ld - X_ld @ w_ols), 10), " ← residuals orthogonal to every column")
print("special case J = 1: fit_ols([1, x_A]) =", np.round(fit_ols(X_ld[:, :2], y_ld), 4),
      " fit_univariate(x_A) =", np.round(fit_univariate(snp_a.astype(float), y_ld), 4))
ols_table(X_ld, y_ld, ["intercept w0", "SNP A (causal)", "SNP B (passenger)"]).round(4)

🔍 **What just happened?** Our normal-equation solver, NumPy's `lstsq` and scikit-learn agree to four decimals, and
$\mathbf X^\top(\mathbf y - \mathbf X\hat{\mathbf w}) = \mathbf 0$. With one SNP the matrix formula reproduces Section 4's
closed form — univariate regression is the special case $J = 1$. And the **puzzle is resolved**: in the joint model SNP A keeps an effect near the
true 0.4, while SNP B's coefficient collapses to about zero with a large $p$-value. Holding A fixed, B adds nothing.

Notice the price: the standard errors of A and B in the joint model are roughly twice what a single-SNP fit would give
(univariate SE ≈ 0.05 in cell 6.1, joint SE ≈ 0.09), because two highly correlated predictors are hard to tell
apart. Strong correlation between predictors (*collinearity*) inflates uncertainty — a theme that returns in Module 07.

### 6.3 Many SNPs at once: the "association strength" picture

Let us fit 200 SNPs of our cohort jointly — the 8 causal ones plus 192 null SNPs — with $N = 500 > J = 200$, and draw the
estimated weight vector as a strip of grey levels — dark for a strong association, white for none.

In [ ]:
# ── 6.5 Fit 200 SNPs jointly and compare with the truth ──
rng_sub = np.random.default_rng(SEED + 6)
null_snps = rng_sub.choice(np.setdiff1d(np.arange(J), causal), 192, replace=False)
subset = np.sort(np.r_[causal, null_snps])                         # 200 SNPs, 8 of them causal
X_sub = np.column_stack([np.ones(N), G[:, subset]])
w_sub = fit_ols(X_sub, y)[1:]                                      # drop the intercept for the picture
is_causal = np.isin(subset, causal)
print(f"N = {N}, J = {len(subset)}; joint estimates: causal SNPs mean |w| = {np.abs(w_sub[is_causal]).mean():.3f}, "
      f"null SNPs mean |w| = {np.abs(w_sub[~is_causal]).mean():.3f}")
print(f"univariate estimates of the same SNPs: causal mean |w| = {np.abs(w_scan[subset][is_causal]).mean():.3f}, "
      f"null mean |w| = {np.abs(w_scan[subset][~is_causal]).mean():.3f}")
print("correlation of joint estimates with univariate estimates for the same SNPs:",
      round(np.corrcoef(w_sub, w_scan[subset])[0, 1], 3))

fig, axes = plt.subplots(2, 1, figsize=(11, 2.4), sharex=True)
for ax, vec, title in zip(axes, [w_true[subset], w_sub], ["true association strengths w (answer key)",
                                                          "estimated by multivariate least squares (N = 500)"]):
    ax.imshow(np.abs(vec)[None, :], cmap="Greys", aspect="auto", vmin=0, vmax=np.abs(w_true).max())
    ax.set_yticks([]); ax.set_title(title, fontsize=9)
axes[1].set_xlabel("SNP (200 loci; darker = larger |w|)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The dark bars of the true $\mathbf w$ reappear in the estimate, but every null SNP also gets a
small non-zero weight — a grey haze of estimation noise (mean $|\hat w|$ about 0.06 for null SNPs, about 0.35 for causal
ones). Because the SNPs in this simulation are independent (no LD), the joint and one-at-a-time estimates tell the same
story (correlation about 0.78; both give null SNPs weights of similar typical size); they differ only through estimation
noise. The joint model really pays off when predictors are correlated, as in the LD puzzle. That grey haze is what the **lasso** of Module 07 will clean up by setting
most weights exactly to zero.

### 6.4 The cost of the closed form

The formula needs $\mathbf X^\top\mathbf X$, a $(J+1)\times(J+1)$ matrix (about $NJ^2$ multiplications to build), and
solving with it costs about $J^3$ operations. Let us time it.

In [ ]:
# ── 6.6 How the cost grows with the number of features J ──
rng_t = np.random.default_rng(SEED + 7)
n_big = 2000
sizes, t_build, t_solve = [100, 200, 400, 800, 1600], [], []
for k in sizes:
    Xk, yk = rng_t.normal(size=(n_big, k)), rng_t.normal(size=n_big)
    t0 = time.perf_counter(); XtX = Xk.T @ Xk; t1 = time.perf_counter()
    np.linalg.solve(XtX, Xk.T @ yk); t2 = time.perf_counter()
    t_build.append(t1 - t0); t_solve.append(t2 - t1)
    print(f"J = {k:5d}: build XᵀX {t_build[-1] * 1e3:7.1f} ms, solve {t_solve[-1] * 1e3:7.1f} ms")
bytes_needed = (3e6) ** 2 * 8
print(f"memory just to store XᵀX for 3 million SNPs: {bytes_needed / 1e12:.0f} terabytes")

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.loglog(sizes, t_build, "o-", label="build XᵀX (≈ N·J²)")
ax.loglog(sizes, t_solve, "s-", label="solve (≈ J³)")
ref = np.array(sizes, float)
ax.loglog(ref, t_solve[-1] * (ref / ref[-1]) ** 3, "k:", label="slope 3 (cubic) reference")
ax.set_xlabel("number of features J"); ax.set_ylabel("seconds"); ax.legend(fontsize=7)
ax.set_title(f"cost of least squares (N = {n_big})")
plt.tight_layout(); plt.show()

🔍 **What just happened?** (Your timings depend on your computer.) Each doubling of $J$ multiplied the solve time by
roughly 3–7 on the machine used to write this notebook, approaching the cubic reference line (a factor of 8 per
doubling) for large $J$; for small $J$ fixed overheads dominate. Extrapolating to
the 3 million SNPs of a modern GWAS is hopeless — storing $\mathbf X^\top\mathbf X$ alone would take 72 terabytes. Real
genome-wide analyses therefore scan SNPs one at a time or use iterative methods that never form $\mathbf X^\top\mathbf X$
(gradient descent, Module 04).

✏️ **Your turn.** In cell 6.1, raise the LD from `0.85` to `0.98` and re-run 6.1–6.4. What happens to the joint
coefficients of A and B and to their standard errors? What would happen at `1.0` (B an exact copy of A)? (Hint: what is
the rank of $\mathbf X$ then?)

### 📘 Lecture notes — multivariate regression and the normal equations

- **Why all SNPs at once.** In non-monogenic diseases several variants act together. Multivariate regression estimates
  the effect of each SNP **in the presence of all the other SNPs**; analyse all features jointly when you can, and fall
  back to one-at-a-time regression when the data are too large.
- **Model.** $y = w_0 + \mathbf x^\top\mathbf w + \epsilon$, $\epsilon\sim\mathcal N(0,\sigma^2)$, with
  $\mathbf x = (x_1,\dots,x_J)^\top$ and $\mathbf w = (w_1,\dots,w_J)^\top$ — one "association strength" per locus;
  a person's predicted trait is the dot product of their genotype row with $\mathbf w$.
- **Augmented data.** Put a 1 on top of $\mathbf x$ and $w_0$ on top of $\mathbf w$: $y = \mathbf x^\top\mathbf w + \epsilon
  = 1\cdot w_0 + x_1w_1 + \dots + x_Jw_J + \epsilon$. Stacking people gives $\mathbf X$ ($N\times(J+1)$, first column all
  ones) and $\mathbf y = \mathbf X\mathbf w + \boldsymbol\epsilon$.
- **MCLE = least squares again:** $\mathbf w_{MCLE} = \arg\min_{\mathbf w}\sum_i(y^i - \mathbf x^{i\top}\mathbf w)^2
  = \arg\min_{\mathbf w}(\mathbf y-\mathbf X\mathbf w)^\top(\mathbf y-\mathbf X\mathbf w)$.
- **Derivation in eight lines.** Expand: $(\mathbf y^\top - \mathbf w^\top\mathbf X^\top)(\mathbf y - \mathbf X\mathbf w)
  = \mathbf y^\top\mathbf y - \mathbf y^\top\mathbf X\mathbf w - \mathbf w^\top\mathbf X^\top\mathbf y + \mathbf w^\top\mathbf X^\top\mathbf X\mathbf w$;
  the two cross terms are equal scalars, so this is $\mathbf y^\top\mathbf y - 2\mathbf w^\top\mathbf X^\top\mathbf y + \mathbf w^\top\mathbf X^\top\mathbf X\mathbf w$.
  Differentiate with $\partial(\mathbf w^\top\mathbf a)/\partial\mathbf w = \mathbf a$ and
  $\partial(\mathbf w^\top\mathbf A\mathbf w)/\partial\mathbf w = (\mathbf A+\mathbf A^\top)\mathbf w$:
  $-2\mathbf X^\top\mathbf y + 2\mathbf X^\top\mathbf X\mathbf w = \mathbf 0$, i.e. the **normal equations**
  $$\mathbf X^\top\mathbf X\,\mathbf w = \mathbf X^\top\mathbf y\qquad\Longrightarrow\qquad \mathbf w_{MCLE} = (\mathbf X^\top\mathbf X)^{-1}\mathbf X^\top\mathbf y .$$
  *In words:* the residuals are orthogonal to every column of $\mathbf X$; one closed-form formula gives all weights.
- **Cost.** Building $\mathbf X^\top\mathbf X$ ($(J+1)\times(J+1)$) costs $\approx NJ^2$ and solving $\approx J^3$ —
  prohibitive for millions of SNPs. Most later models (logistic regression, Module 09) have no closed form at all and
  need iterative optimisation.
- **In code:** solve the linear system (`np.linalg.solve`), never form the inverse.

> 🎓 **Exam tip.** Reproduce the derivation: expand, combine the two cross terms (*why* are they equal?), differentiate with the two matrix rules, use symmetry of $\mathbf X^\top\mathbf X$. Know the size of $\mathbf X^\top\mathbf X$: $(J+1)\times(J+1)$, independent of $N$. A common slip is to write $[-(y^i-\mathbf x^{i\top}\mathbf w)]^2$ with the minus sign inside the square — it belongs outside.

---
## 7 · A clinical example: prostate cancer

> 📍 **Where we are.** We have a complete, verified least-squares toolkit. The 02-620 lecture applies it to a classic
> clinical data set from the textbook *The Elements of Statistical Learning* (ESL, Hastie, Tibshirani & Friedman).

**The study.** Stamey *et al.* (1989) measured **97 men** who were about to undergo a radical prostatectomy (removal of the
prostate for cancer) and asked: *is the level of prostate-specific antigen correlated with a number of clinical
measures?* PSA is a protein made by the prostate and measured in blood; it is widely used to monitor prostate cancer.
The outcome is $y$ = `lpsa` = log(PSA). The eight predictors $x_1,\dots,x_8$:

| column | meaning |
|---|---|
| `lcavol` | log cancer volume |
| `lweight` | log prostate weight |
| `age` | age in years |
| `lbph` | log amount of benign prostatic hyperplasia (non-cancerous enlargement) |
| `svi` | seminal vesicle invasion (0 = no, 1 = yes) |
| `lcp` | log capsular penetration (how far the tumour has grown through the prostate capsule) |
| `gleason` | Gleason score (pathologist's grade of how abnormal the tumour tissue looks; 6–9 here) |
| `pgg45` | percentage of Gleason scores 4 or 5 |

So the data are a $97\times8$ matrix $\mathbf X$ and a vector $\mathbf y$ — exactly the layout of a GWAS, with clinical
measures instead of SNPs. ESL splits the men into **67 training** and **30 test** men. The lecture follows the pipeline
every analysis should follow: **obtain the data → visualise each predictor against $y$ → fit the model → check that the
fitted coefficients agree with what the plots suggested.**

**What the original data gave.** ESL standardised the predictors, fitted least squares on the 67 training men and
reported (Table 3.2):

| term | intercept | lcavol | lweight | age | lbph | svi | lcp | gleason | pgg45 |
|---|---|---|---|---|---|---|---|---|---|
| coefficient | 2.46 | 0.68 | 0.26 | −0.14 | 0.21 | 0.31 | −0.29 | −0.02 | 0.27 |
| Z-score ($t$) | 27.60 | 5.37 | 2.75 | −1.40 | 2.06 | 2.47 | −1.87 | −0.15 | 1.74 |

Log cancer volume has by far the largest coefficient — matching the scatter plots, in which cancer volume showed the
clearest trend with PSA. On the 30 test men the fitted model had a mean squared prediction error of **0.521**, against
**1.057** for simply predicting the training mean (ESL Table 3.3).

**Our data.** We use `simulate_prostate_clinical()`: 97 simulated men whose predictors have the ranges, floors and
correlations of the original study, and whose `lpsa` was generated from the standardised predictors with **the
coefficients above** (intercept 2.47) plus Gaussian noise (SD 0.71). So here, unlike in the original study, we know the true coefficients
and can check how well 67 men recover them. The numbers will not reproduce the paper — a different sample of 97 men
never would — but the lessons do.

In [ ]:
# ── 7.1 Simulate the prostate-cancer study (SIMULATED data shaped like Stamey et al. 1989) ──
prostate = simulate_prostate_clinical()
predictors = ["lcavol", "lweight", "age", "lbph", "svi", "lcp", "gleason", "pgg45"]
is_train = prostate["train"].to_numpy()
original = pd.Series([2.46, 0.68, 0.26, -0.14, 0.21, 0.31, -0.29, -0.02, 0.27],
                     index=["intercept"] + predictors, name="original study (ESL Table 3.2)")
print("source: SIMULATED clinical data shaped like the prostate-cancer study of Stamey et al. (1989)")
print(f"{len(prostate)} men: {is_train.sum()} training, {(~is_train).sum()} test")
prostate.describe().loc[["mean", "std", "min", "max"]].round(2)

In [ ]:
# ── 7.2 Look first: lpsa against each clinical measure ──
fig, axes = plt.subplots(2, 4, figsize=(11, 4.6), sharey=True)
for ax, col in zip(axes.ravel(), predictors):
    r = np.corrcoef(prostate[col], prostate["lpsa"])[0, 1]
    ax.scatter(prostate[col], prostate["lpsa"], s=10, alpha=0.6, color="tab:purple")
    ax.set_title(f"{col}  (r = {r:+.2f})"); ax.set_xlabel(col)
for ax in axes[:, 0]:
    ax.set_ylabel("lpsa = log(PSA)")
fig.suptitle("Prostate cancer data (SIMULATED, shaped like Stamey et al. 1989; 97 men)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** As in the original study, log cancer volume shows the clearest trend with log PSA
($r \approx 0.76$), followed by `pgg45`, `lcp` and `lweight`; `age` shows little. `svi` is binary and `gleason` takes only
four values, so their scatter plots are columns of points; in `lbph` and `lcp` many men share the smallest value,
−1.39 = log 0.25 — the original data used this floor when the measured amount was zero or tiny, and the simulator copies
it. Looking before modelling already suggests which predictors matter.

**Standardising the predictors.** The predictors have very different units (years, logs, percentages), so ESL first
**standardises** each column to mean 0 and standard deviation 1, $z = (x - \text{mean})/\text{SD}$. A coefficient then
means "change in log PSA per one-SD increase of that predictor", which makes coefficients comparable. ESL computed the
means and SDs from **all 97 men**; we do the same first, so that our table is directly comparable with the original one.

In [ ]:
# ── 7.3 OLS on the 67 training men, predictors standardised with all 97 men (as ESL did) ──
X_raw = prostate[predictors].to_numpy(float)
y_psa = prostate["lpsa"].to_numpy()
Z_all, _, _ = standardize(X_raw)                                   # mean/SD of all 97 men (as ESL does)
A_all = np.column_stack([np.ones(len(y_psa)), Z_all])
tab = ols_table(A_all[is_train], y_psa[is_train], ["intercept"] + predictors)
tab["original study"] = original
tab["truth (simulation)"] = original.where(original.index != "intercept", 2.47)   # what generated our data
display_tab = tab.round({"coef": 3, "SE": 3, "t": 2, "p": 4})
w_psa = tab["coef"].to_numpy()
mse_test = np.mean((y_psa[~is_train] - A_all[~is_train] @ w_psa) ** 2)
mse_base = np.mean((y_psa[~is_train] - y_psa[is_train].mean()) ** 2)   # predict every test man by the training mean
z_dev = (tab["coef"] - tab["truth (simulation)"]) / tab["SE"]          # distance from the truth in SE units
print(f"test MSE of least squares: {mse_test:.3f}   |   test MSE of 'predict the training mean': {mse_base:.3f}")
print(f"🧪 ground truth: {np.sum(np.abs(z_dev) < 2)} of 9 true coefficients lie within 2 SE of our estimates; "
      f"largest deviation: {z_dev.abs().idxmax()} ({z_dev.abs().max():.2f} SE)")
print(f"mean lpsa: training men {y_psa[is_train].mean():.2f}, test men {y_psa[~is_train].mean():.2f}")
display_tab

🔍 **What just happened?** The fit on 67 simulated men gives `lcavol` 0.72 (truth 0.68), and most other coefficients
land near the values that generated the data — `lweight` 0.41 vs 0.26, `age` −0.24 vs −0.14, `lcp` −0.36 vs −0.29,
`pgg45` 0.47 vs 0.27. 🧪 Because we know the truth, we can check the standard errors: an estimate should lie within two
SEs of the truth about 95 % of the time, and 8 of the 9 do. The ninth is the intercept (2.32 vs the true 2.47, 2.02 SE
away): the 67 training men happen to have a lower average log PSA (2.25) than the 30 test men (2.66). The differences
are pure sampling noise from 67 men and noise SD 0.71 — exactly the uncertainty a real study lives with, invisible
there because the truth is unknown. The test error, **0.348**, is about a third of the error of simply predicting the
training mean (1.153) — the same picture as in the original study (0.521 vs 1.057), with numbers that differ because the
sample differs.

⚠️ **A subtle leak.** Standardising with all 97 men lets the test men's means and SDs influence the training data. The
clean way is to compute means and SDs from the **training men only** and apply them to the test men. Does it matter here?

In [ ]:
# ── 7.4 Standardise with the training men only — and with no standardisation at all ──
Z_tr, mu_tr, sd_tr = standardize(X_raw[is_train])                 # statistics of the 67 training men
Z_te, _, _ = standardize(X_raw[~is_train], mu_tr, sd_tr)           # … applied unchanged to the test men
w_tr = fit_ols(np.column_stack([np.ones(len(Z_tr)), Z_tr]), y_psa[is_train])
pred_tr = np.column_stack([np.ones(len(Z_te)), Z_te]) @ w_tr
w_raw = fit_ols(np.column_stack([np.ones(len(y_psa)), X_raw])[is_train], y_psa[is_train])
pred_raw = np.column_stack([np.ones(len(y_psa)), X_raw])[~is_train] @ w_raw
pred_all = A_all[~is_train] @ w_psa
pd.DataFrame({"std. with all 97": w_psa, "std. with 67 training": w_tr, "raw units": w_raw},
             index=["intercept"] + predictors).round(3).T.assign(
    test_MSE=[mse_test, np.mean((y_psa[~is_train] - pred_tr) ** 2), np.mean((y_psa[~is_train] - pred_raw) ** 2)])

🔍 **What just happened?** The coefficients change — slightly between the two standardisations (`lcavol` 0.72 → 0.77,
intercept 2.32 → 2.25), completely in raw units — yet the **test MSE is identical** in all three versions (0.348).
Shifting a column by a constant or multiplying it by a positive number only *reparameterises* the same family of linear
predictions: the intercept absorbs the shift and the coefficient absorbs the scale, so the least-squares *predictions* are
unchanged. For OLS the leak above is therefore harmless.

⚠️ This invariance is special to unpenalised least squares. Ridge and lasso (Module 07) penalise the *size* of the
coefficients, which depends on the units — there, standardisation (done with training data only) matters.

### 7.1 How good is the fit, and what does it mean?

In [ ]:
# ── 7.5 R², residuals and predictions for new patients ──
fitted_tr = A_all[is_train] @ w_psa
resid_tr = y_psa[is_train] - fitted_tr
r2_train = 1 - np.sum(resid_tr ** 2) / np.sum((y_psa[is_train] - y_psa[is_train].mean()) ** 2)
r2_test = 1 - np.sum((y_psa[~is_train] - pred_all) ** 2) / np.sum((y_psa[~is_train] - y_psa[is_train].mean()) ** 2)
print(f"R² on the 67 training men: {r2_train:.3f}   |   R² on the 30 test men: {r2_test:.3f}")
print(f"correlation of lcp with lpsa (training men): {np.corrcoef(X_raw[is_train, 5], y_psa[is_train])[0, 1]:+.2f}, "
      f"but its multivariate coefficient is {w_psa[6]:+.2f}")
print(f"correlation of lcp with lcavol: {np.corrcoef(X_raw[is_train, 5], X_raw[is_train, 0])[0, 1]:.2f}, "
      f"with pgg45: {np.corrcoef(X_raw[is_train, 5], X_raw[is_train, 7])[0, 1]:.2f}, "
      f"with svi: {np.corrcoef(X_raw[is_train, 5], X_raw[is_train, 4])[0, 1]:.2f};  "
      f"correlation of gleason with pgg45: {np.corrcoef(X_raw[is_train, 6], X_raw[is_train, 7])[0, 1]:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.3))
axes[0].scatter(fitted_tr, resid_tr, s=12, color="tab:purple"); axes[0].axhline(0, color="k", lw=0.8)
axes[0].set_xlabel("fitted lpsa (training men)"); axes[0].set_ylabel("residual"); axes[0].set_title("residuals vs fitted: no clear pattern")
axes[1].scatter(y_psa[~is_train], pred_all, s=14, color="tab:orange")
lims = [y_psa.min() - 0.3, y_psa.max() + 0.3]
axes[1].plot(lims, lims, "k--", lw=1, label="perfect prediction")
axes[1].set_xlabel("observed lpsa (30 test men)"); axes[1].set_ylabel("predicted lpsa"); axes[1].legend(fontsize=7)
axes[1].set_title("new patients: predicted vs observed")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The model explains about 80 % of the variance of log PSA in the training men but only about
70 % in the 30 test men — a fit always looks better on the data it was trained on. The residuals show no obvious pattern, so a linear model is
a reasonable first description — as it should be, since the data were generated by one.

**Interpreting coefficients — carefully.**
* `lcavol` has by far the largest coefficient and $t$-value: larger tumours go with higher PSA, as the scatter plots
  suggested — the "check that the model agrees with the plots" step of the pipeline.
* `lcp` correlates **positively** with log PSA on its own ($r\approx +0.55$ in the training men), but its multivariate
  coefficient is **negative** (−0.36; in the original study −0.29). There is no contradiction: a multivariate coefficient
  answers "what is the association of `lcp` *at fixed values of all the other predictors*?" — and `lcp` is strongly
  correlated with `lcavol` (0.64) and `pgg45` (0.71). Once those are known, a higher `lcp` goes with a slightly
  *lower* PSA. In our simulation this conditional effect is real (the data were generated with −0.29, $t\approx-2.9$
  here); in the original study it was uncertain ($t = -1.87$). Either way, the marginal and the conditional association
  answer different questions — the LD puzzle of Section 6 in a clinical setting.
* `gleason` gets a coefficient near zero although its correlation with log PSA is positive: its information is largely
  contained in `pgg45` (correlation 0.82).
* These are **associations in 67 men**, not causal effects. A coefficient says nothing about what would happen if we
  *changed* a predictor.

✏️ **Your turn.** Remove `lcavol` from the predictors (use `A_all[:, [0, 2, 3, 4, 5, 6, 7, 8]]`) and refit with `ols_table`.
What happens to the test MSE and to the coefficients of `lcp` and `svi`? Explain with the correlation table
`prostate[predictors].corr()`.

### 📘 Lecture notes — the prostate-cancer example

- **Question.** Is the level of prostate-specific antigen correlated with clinical measures in 97 men about to receive
  a radical prostatectomy? $y$ = log PSA; $\mathbf x$ = eight clinical measures (log cancer volume `lcavol`, log
  prostate weight `lweight`, `age`, log benign prostatic hyperplasia `lbph`, seminal vesicle invasion `svi`, log
  capsular penetration `lcp`, Gleason score `gleason`, % Gleason scores 4 or 5 `pgg45`). Source: ESL (Hastie,
  Tibshirani & Friedman), data of Stamey *et al.* (1989).
- **Pipeline.** Obtain the data → plot $y$ against each $x_j$ (here: cancer volume shows the strongest trend) → fit
  $\mathbf w = (\mathbf X^\top\mathbf X)^{-1}\mathbf X^\top\mathbf y$ → check that the coefficients agree with the plots.
- **Result on the original data** (67 training men, standardised predictors): intercept 2.46, lcavol 0.68, lweight 0.26,
  age −0.14, lbph 0.21, svi 0.31, lcp −0.29, gleason −0.02, pgg45 0.27; test error 0.521 vs 1.057 for the mean.
  *In words:* one SD more log cancer volume goes with +0.68 log PSA, holding the other measures fixed; the intercept is
  the predicted log PSA of an average man.
- **Reading coefficients.** They are per SD of each (standardised) predictor and conditional on all other predictors —
  so a coefficient can have the opposite sign of the marginal correlation (`lcp`) or vanish when another predictor
  carries the same information (`gleason` vs `pgg45`). Associations, not causes.
- **Practical points.** Standardise with training statistics only (for OLS the predictions do not change; for ridge and
  lasso they do); judge the model on test men, not on the training men.

> 🎓 **Exam tip.** Remember that $y$ is **log** PSA and the predictors are standardised, so "0.68" means +0.68 in log PSA per SD of log cancer volume. Be ready to explain why a coefficient can have the opposite sign of the marginal correlation.

---
## 8 · Limitations of MLE: more SNPs than people

> 📍 **Where we are.** *Question: do you see any limitations of the MLE for multivariate linear regression?* The good
> news is a closed-form estimate. We met one limitation already (cost, Section 6.4). The decisive one:
> $\mathbf X^\top\mathbf X$ **is not invertible when the number of samples $N$ is smaller than the number of inputs
> $J$** — consider 3 million SNPs and 1,000 individuals.

Why? The rank of a matrix — the number of truly independent columns (Module 03) — can never exceed its number of rows. An
$N\times(J+1)$ matrix $\mathbf X$ with $N < J+1$ has rank at most $N$, and so does the $(J+1)\times(J+1)$ matrix
$\mathbf X^\top\mathbf X$. A square matrix with rank below its size has at least one zero eigenvalue and **no inverse**:
the normal equations (7) then have infinitely many solutions.

In [ ]:
# ── 8.1 100 people, 2,000 SNPs: the normal equations break ──
rng8 = np.random.default_rng(SEED + 8)
G8, _, _ = simulate_genotypes(rng8, 600, 2000)
y8, w8_true, causal8 = simulate_quantitative_trait(rng8, G8, n_causal=8, h2=0.4)
train8, test8 = np.arange(100), np.arange(100, 600)               # 100 training people, 500 new people for testing
X8 = np.column_stack([np.ones(600), G8.astype(float)])
XtX8 = X8[train8].T @ X8[train8]
eig = np.linalg.eigvalsh(XtX8)                                     # eigenvalues of the 2001 × 2001 matrix
print(f"X has shape {X8[train8].shape}; rank of X = {np.linalg.matrix_rank(X8[train8])}; XᵀX has shape {XtX8.shape}")
print(f"eigenvalues of XᵀX that are (numerically) zero: {np.sum(eig < 1e-8 * eig.max())} of {len(eig)}")
try:
    w8 = np.linalg.solve(XtX8, X8[train8].T @ y8[train8])
    print("np.linalg.solve did NOT complain and returned a 'solution':")
    print(f"   largest |w| = {np.abs(w8).max():.2f}; training MSE = {np.mean((y8[train8] - X8[train8] @ w8) ** 2):.2e}; "
          f"test MSE = {np.mean((y8[test8] - X8[test8] @ w8) ** 2):.2f}")
except np.linalg.LinAlgError as err:
    print("np.linalg.solve raised LinAlgError:", err)
print(f"for comparison, test MSE of 'predict the training mean' = {np.mean((y8[test8] - y8[train8].mean()) ** 2):.2f}")

🔍 **What just happened?** $\mathbf X$ has rank 100, so the $2001\times2001$ matrix $\mathbf X^\top\mathbf X$ has 1,901
zero eigenvalues — it is singular. Depending on your NumPy/LAPACK build, `np.linalg.solve` either raises `LinAlgError` or —
as happened when this notebook was written — silently returns a "solution" computed from round-off errors. That solution
fits the 100 training people *perfectly* (training MSE ≈ $10^{-28}$, i.e. zero) and predicts new people dreadfully: its
test MSE is many times worse than predicting everyone by the average. **A perfect training fit is not a good model.**

Infinitely many weight vectors fit the training data perfectly. Here are two of them: the "smallest" one (what `lstsq`
returns) and the same vector plus a direction that $\mathbf X$ cannot see.

In [ ]:
# ── 8.2 Infinitely many perfect fits ──
w_min = np.linalg.lstsq(X8[train8], y8[train8], rcond=None)[0]     # the minimum-length perfect fit
_, _, Vt = np.linalg.svd(X8[train8], full_matrices=True)
v_null = Vt[-1]                                                    # a direction with X v = 0 (the "null space")
w_other = w_min + 5 * v_null                                       # another perfect fit
for name, w in [("minimum-norm w", w_min), ("w + 5·v (v in null space)", w_other)]:
    print(f"{name:27s}: training MSE {np.mean((y8[train8] - X8[train8] @ w) ** 2):.1e}, "
          f"test MSE {np.mean((y8[test8] - X8[test8] @ w) ** 2):6.2f}")
print("|X v| for the null-space direction:", f"{np.linalg.norm(X8[train8] @ v_null):.1e}")

🔍 **What just happened?** Both weight vectors reproduce the 100 training phenotypes exactly — the data cannot distinguish
them — yet they predict new people differently, and neither predicts better than the average. The data alone do not
determine $\mathbf w$ when $N<J$; we need **extra information** about which $\mathbf w$ is plausible. That is precisely what a
prior provides (MAP estimation, Module 07).

The trouble starts well before $J$ reaches $N$. Let us add SNPs one block at a time (the 8 causal ones first, then null
SNPs) with the same 100 training people.

In [ ]:
# ── 8.3 Training and test error as the number of SNPs grows (N = 100 fixed) ──
order = np.r_[causal8, np.setdiff1d(np.arange(2000), causal8)]   # causal SNPs first (the best case), then null SNPs
Js = [1, 2, 4, 8, 15, 25, 40, 60, 75, 85, 92, 96, 98]
mse_tr, mse_te = [], []
for k in Js:
    Xk = np.column_stack([np.ones(600), G8[:, order[:k]]])
    wk = np.linalg.lstsq(Xk[train8], y8[train8], rcond=None)[0]
    mse_tr.append(np.mean((y8[train8] - Xk[train8] @ wk) ** 2))
    mse_te.append(np.mean((y8[test8] - Xk[test8] @ wk) ** 2))
print("J        :", Js)
print("train MSE:", np.round(mse_tr, 2))
print("test MSE :", np.round(mse_te, 2))

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(Js, mse_tr, "o-", label="training error (100 people)")
ax.plot(Js, mse_te, "s-", label="test error (500 new people)")
ax.axvline(8, color="0.6", ls=":", lw=1); ax.text(8.5, 0.15, "all 8 causal\nSNPs included", fontsize=7)
ax.set_yscale("log"); ax.set_xlabel("number of SNPs J in the model"); ax.set_ylabel("mean squared error")
ax.set_title("least squares with N = 100: more SNPs → overfitting"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The training error falls steadily towards zero as SNPs are added — every extra column gives
least squares more freedom to chase the noise of these particular 100 people. The test error falls only while *causal*
SNPs enter (up to $J = 8$), then climbs, and explodes as $J$ approaches $N = 100$. This is **overfitting**: the model
memorises the training sample instead of learning the biology. It is the central problem of statistical genetics and of
machine learning with many features; Module 23 studies it in depth.

✏️ **Your turn.** Change the training set to 300 people (`train8 = np.arange(300)` and `test8 = np.arange(300, 600)`) and re-run
cells 8.1–8.3 (extend `Js` up to 290). Where does the test error now start to rise?

**What do we do if $N<J$?** Use a **MAP estimate**. A Gaussian prior on the weights adds
$\lambda\mathbf I$ to $\mathbf X^\top\mathbf X$ — and you saw in Module 03 that adding $\lambda$ to the diagonal raises every
eigenvalue by $\lambda$, so the matrix becomes invertible. That is ridge regression, and with an $L_1$ penalty it becomes the
lasso, which can set most of the 2,000 weights exactly to zero. Both are the subject of
[Module 07](07_ridge_lasso_sparse_regression.ipynb).

### 📘 Lecture notes — limitations of the maximum-likelihood estimate

- **Good news:** least squares has a closed-form MLE, $\mathbf w_{MCLE} = (\mathbf X^\top\mathbf X)^{-1}\mathbf X^\top\mathbf y$.
- **Bad news 1 — cost:** forming and inverting the $(J+1)\times(J+1)$ matrix $\mathbf X^\top\mathbf X$ is expensive for
  large $J$ ($\approx J^3$ operations).
- **Bad news 2 — invertibility:** $(\mathbf X^\top\mathbf X)^{-1}$ does not exist when the number of samples $N$ is
  smaller than the number of inputs $J$ (think of 3 million SNPs and 1,000 individuals). Reason:
  $$\operatorname{rank}(\mathbf X^\top\mathbf X) = \operatorname{rank}(\mathbf X) \le N < J+1 .$$
  *In words:* a "fat" matrix (more columns than rows) cannot have more independent columns than rows, so
  $\mathbf X^\top\mathbf X$ has zero eigenvalues (determinant 0) and the normal equations have infinitely many solutions.
- **Bad news 3 — overfitting:** every solution then fits the training data perfectly and predicts new data badly; the
  test error starts to rise long before $J$ reaches $N$.
- **What do we do if $N<J$?** Use a **MAP estimate**: a Gaussian prior on $\mathbf w$ adds $\lambda\mathbf I$ to
  $\mathbf X^\top\mathbf X$ (ridge regression), which is invertible for any $\lambda > 0$ — Module 07.

> 🎓 **Exam tip.** Say *why* $\mathbf X^\top\mathbf X$ is singular when $N<J$ (the rank argument), and remember the other two limitations: cost ($J^3$) and overfitting even when $N$ is only slightly larger than $J$.

## 📌 Key takeaways

- **Regression** is supervised learning with a real-valued output; in a GWAS the output is a quantitative phenotype and the
  inputs are 0/1/2 genotypes. We model $P(Y\mid X)$ with the two-step recipe: choose a parameterised model, then estimate
  by MLE or MAP.
- **GWAS** genotype unrelated individuals and test every SNP; the result is shown as a **Manhattan plot** of
  $-\log_{10}p$ along the genome. Discrete phenotypes → classification, real-valued phenotypes → regression.
- **Univariate model:** $y = w_0 + wx + \epsilon$, $\epsilon\sim\mathcal N(0,\sigma^2)$, i.e. $p(y\mid x) = \mathcal N(w_0 + wx, \sigma^2)$.
- **MLE for least squared error:** maximising the conditional likelihood under Gaussian noise *is* minimising
  $\sum_i(y^i - \hat y^i)^2$ — MLE gives least squares its probabilistic meaning — with closed form $w = \operatorname{cov}(x,y)/\operatorname{var}(x)$, $w_0 = \bar y - w\bar x$.
- **A GWAS** scans SNPs one at a time; standard errors, $t$-statistics, $p$-values and a Bonferroni threshold turn slopes into
  hits — which are **associations** (LD, population structure), not proof of causation.
- **Multivariate regression** estimates each SNP's effect in the presence of the others; with the augmented $\mathbf X$, the
  normal equations $\mathbf X^\top\mathbf X\mathbf w = \mathbf X^\top\mathbf y$ give $\mathbf w = (\mathbf X^\top\mathbf X)^{-1}\mathbf X^\top\mathbf y$
  (solve, don't invert).
- **Prostate cancer:** in the original study least squares on 67 men gave `lcavol` the largest coefficient (0.68) and
  test MSE 0.521 (vs 1.057 for the mean); on our simulated look-alike the fit recovers the true coefficients within their
  standard errors (test MSE 0.348 vs 1.153). Multivariate coefficients are conditional on the other predictors (the sign
  of `lcp` flips), and OLS predictions do not depend on how the predictors are scaled.
- **Limitations of MLE:** cost $\propto J^3$, and when $N<J$ the matrix $\mathbf X^\top\mathbf X$ is singular — infinitely many
  perfect but useless fits. Overfitting starts well before $J = N$. → MAP / ridge / lasso (Module 07).

---
## ✅ Checkpoint

1. Three people have genotypes $x = (0, 1, 2)$ and phenotypes $y = (1, 2, 6)$. Compute $\hat w$ and $\hat w_0$ by hand.
   <details><summary>Answer</summary>$\bar x = 1$, $\bar y = 3$. $\sum(x-\bar x)(y-\bar y) = (-1)(-2) + 0\cdot(-1) + (1)(3) = 5$; $\sum(x-\bar x)^2 = 2$. So $\hat w = 5/2 = 2.5$ and $\hat w_0 = 3 - 2.5\cdot1 = 0.5$. Check in code: `fit_univariate(np.array([0., 1, 2]), np.array([1., 2, 6]))`.</details>
2. In the derivation (1)→(6), why may we drop the term $\log\frac{1}{\sqrt{2\pi\sigma^2}}$ and the factor $\frac{1}{\sigma^2}$?
   <details><summary>Answer</summary>The first is the same constant for every candidate $(w_0, w)$, so adding it does not change where the maximum is. The second is a positive constant multiplying the whole objective; scaling by a positive number does not move the maximiser (a negative one would turn max into min).</details>
3. Why does $P(x^i\mid\theta)$ disappear from the likelihood of a regression model?
   <details><summary>Answer</summary>The regression model only describes $P(y\mid x)$; the parameters $w_0, w$ do not appear in the distribution of the genotypes. So $\prod_i P(x^i)$ is a constant with respect to $w_0, w$ and does not affect the arg max — we maximise the *conditional* likelihood.</details>
4. A SNP has $\hat w = 0.30$ and $\operatorname{SE} = 0.06$ in a GWAS of 1 million SNPs. Is it genome-wide significant?
   <details><summary>Answer</summary>$t = 0.30/0.06 = 5$. For large $N$, $p \approx 2\,P(Z > 5) \approx 5.7\times10^{-7}$ (`2 * stats.norm.sf(5)`). The Bonferroni threshold is $0.05/10^6 = 5\times10^{-8}$, so **no** — suggestive, but not genome-wide significant.</details>
5. In step (4) of the normal-equation derivation, why is $\mathbf y^\top\mathbf X\mathbf w = \mathbf w^\top\mathbf X^\top\mathbf y$?
   <details><summary>Answer</summary>$\mathbf y^\top\mathbf X\mathbf w$ is $1\times N$ times $N\times(J+1)$ times $(J+1)\times1$ = a single number. A number equals its own transpose, and $(\mathbf y^\top\mathbf X\mathbf w)^\top = \mathbf w^\top\mathbf X^\top\mathbf y$ by the reverse-order rule. So the two cross terms combine into $-2\mathbf w^\top\mathbf X^\top\mathbf y$.</details>
6. Code reading: what does `np.linalg.solve(X.T @ X, X.T @ y)` compute, what must `X` contain, and why not `np.linalg.inv`?
   <details><summary>Answer</summary>It solves the normal equations $\mathbf X^\top\mathbf X\mathbf w = \mathbf X^\top\mathbf y$ and returns the least-squares weights. `X` must be the augmented matrix (first column all ones) if we want an intercept. Solving is faster and numerically more accurate than forming the inverse and multiplying.</details>
7. Standardising the prostate predictors with training-only means and SDs changed the coefficients but not the test MSE. Why? Would the same hold for ridge regression?
   <details><summary>Answer</summary>Shifting and positively rescaling columns gives the same set of possible linear predictions (the intercept absorbs shifts, coefficients absorb scales), so the least-squares predictions are identical. Ridge penalises $\sum_j w_j^2$, which depends on the scale of each column, so its predictions do change with standardisation.</details>
8. With $N = 1{,}000$ people and $J = 3{,}000{,}000$ SNPs, what is the largest possible rank of $\mathbf X^\top\mathbf X$, and what does that imply?
   <details><summary>Answer</summary>$\operatorname{rank}(\mathbf X^\top\mathbf X) = \operatorname{rank}(\mathbf X) \le 1{,}000$, while the matrix is $3{,}000{,}001\times3{,}000{,}001$. It is singular, so $(\mathbf X^\top\mathbf X)^{-1}$ does not exist and infinitely many $\mathbf w$ fit the data perfectly — we need a prior (MAP/ridge/lasso).</details>

## 🧠 Questions & discussion

1. In real cohorts the phenotype spread sometimes looks larger for `AA` than for `TT`, yet the model uses one $\sigma^2$. How would you check this assumption on the cohort of Section 2, and how could the model be changed?
2. GWAS now include more than a million people. What does a larger $N$ buy you (think of $\operatorname{SE}(\hat w)$ and power), and which problems does it *not* solve (think of Section 5.5)?
3. The rule of thumb is to analyse all features together when you can, but fall back to univariate regression when the data are too large. Give two other reasons why GWAS still mostly report one-SNP-at-a-time tests.
4. For a case–control study, $y\in\{0,1\}$. What goes wrong if you fit $y = w_0 + wx + \epsilon$ with Gaussian noise? (Preview of Module 09, logistic regression.)
5. In the prostate data `lcp` has a negative coefficient. A clinician concludes that capsular penetration *lowers* PSA. What would you tell them?

## 🚀 Bonus — going further

- **Covariates against confounding.** In Section 5.5, instead of centring within populations, run the multivariate regression $y = w_0 + w\,x_j + v\cdot\text{pop} + \epsilon$ for every SNP with `fit_ols` and compare the hits. Real GWAS use the top principal components of the genotype matrix instead of a known population label (Module 14; Price *et al.* 2006).
- **Gradient descent for least squares.** Minimise $(\mathbf y-\mathbf X\mathbf w)^\top(\mathbf y-\mathbf X\mathbf w)$ on the prostate data with the gradient of equation (6) and a fixed step size (Module 04). How many iterations until you match `fit_ols` to three decimals? Why does standardising the predictors speed it up?
- **A numerically better solver.** Compute the QR decomposition `Q, R = np.linalg.qr(X)` and solve $\mathbf R\mathbf w = \mathbf Q^\top\mathbf y$. Compare with `fit_ols` when two columns are almost identical (forming $\mathbf X^\top\mathbf X$ squares the condition number).
- **Deep dive in this repository:** [`../../RegularizedLinearRegression/`](../../RegularizedLinearRegression/) shows the failure of OLS when $p\gg n$ on simulated pathways and on a real wheat genomic-prediction data set (1,279 markers), then fixes it with ridge and lasso — the natural sequel to Section 8.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module 06 — Linear regression and genome-wide association            date:
Input: SIMULATED GWAS cohort (500 people × 2,000 SNPs, 8 causal, h² = 0.4); SIMULATED prostate data (97 men, 8 predictors)
Output: univariate fits, a Manhattan plot, multivariate OLS coefficients, prostate test MSE
Method: maximum conditional likelihood under Gaussian noise = least squares; t-tests; normal equations
Key equation: w = (XᵀX)⁻¹Xᵀy   (univariate: w = cov(x,y)/var(x), w0 = ȳ − w·x̄)
Key parameters: N, J, heritability h², Bonferroni level 0.05/J, train/test split (67/30)
What changed when I modified … (h², N, LD strength, number of SNPs in the model):
One failure mode: N < J → XᵀX singular → infinitely many perfect, useless fits
Connection to the 02-620 topics (which idea of the lecture this illustrates):
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists* (course design S. Kim; instructors M. Xu and M. Zhang) — this module teaches the material of its lecture on regression (part 1).
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — §3.1–3.2 (least squares; prostate example, Tables 3.1–3.3).
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — Chapter 7 (linear regression, MLE).
- Stamey T. A. *et al.* (1989) Prostate specific antigen in the diagnosis and treatment of adenocarcinoma of the prostate. II. Radical prostatectomy treated patients. *J Urol* 141:1076–1083.
- Turnbull C. *et al.* (2010) Genome-wide association study identifies five new breast cancer susceptibility loci. *Nat Genet* 42:504–507 (the breast-cancer GWAS described in Section 1).
- Price A. L. *et al.* (2006) Principal components analysis corrects for stratification in genome-wide association studies. *Nat Genet* 38:904–909.
- Visscher P. M. *et al.* (2017) 10 years of GWAS discovery: biology, function, and translation. *Am J Hum Genet* 101:5–22.
- Petersen K. B., Pedersen M. S. *The Matrix Cookbook* (https://www.math.uwaterloo.ca/~hwolkowi/matrixcookbook.pdf), eqs. 69 and 81 (the two derivative rules of Section 6).

## ➡️ Next

Least squares cannot cope with $N<J$. In [Module 07 · 🪢 Regularization: ridge, lasso and sparse genetic effects](07_ridge_lasso_sparse_regression.ipynb) we put a prior on the weights, derive ridge regression as a MAP estimate, and then use the lasso to pick out the few SNPs that matter.